In [1]:

from pathlib import Path
import shutil
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

print("=" * 75)
print("SCR-CMX: DISK AND ARCHITECTURE INSPECTION")
print("=" * 75)

# 1. Disk space check
usage = shutil.disk_usage(SCR_DIR)
print("\n--- Disk usage ---")
print(f"Total:     {usage.total / 1024**3:.2f} GB")
print(f"Used:      {usage.used / 1024**3:.2f} GB")
print(f"Free:      {usage.free / 1024**3:.2f} GB")

if usage.free / 1024**3 < 20:
    print("WARNING: Less than 20 GB free. Avoid another full checkpoint copy.")

# 2. Check important files in the SCR copy
files_to_inspect = [
    "models/net_utils_e5.py",
    "models/encoders/dual_segformer_e5.py",
    "models/builder.py",
    "train.py",
    "config.py",
]

for relative_path in files_to_inspect:
    path = SCR_DIR / relative_path
    print(f"\n{'=' * 75}\nFILE: {relative_path}")
    if not path.exists():
        print("FILE MISSING")
        continue

    lines = path.read_text(errors="replace").splitlines()
    print(f"Total lines: {len(lines)}")

    # Locate architecture and loss-related definitions
    patterns = re.compile(
        r"^\s*(class |def |    def |    class )|"
        r"reliability|counterfactual|FeatureFusion|FeatureRectify|"
        r"backbone|optimizer|criterion|loss"
    )

    matches = [
        (i, line) for i, line in enumerate(lines)
        if patterns.search(line)
    ]

    # Print matches with nearby context, merging overlapping ranges
    ranges = []
    for i, _ in matches:
        start, end = max(0, i - 2), min(len(lines), i + 4)
        if ranges and start <= ranges[-1][1]:
            ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
        else:
            ranges.append((start, end))

    # Limit output so the notebook does not become enormous
    printed = 0
    max_lines = 240 if "net_utils" in relative_path else 200

    for start, end in ranges:
        chunk_len = end - start
        if printed + chunk_len > max_lines:
            print("... output limit reached; remaining sections omitted ...")
            break

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")
        printed += chunk_len

print("\nInspection complete. No source files were modified.")


SCR-CMX: DISK AND ARCHITECTURE INSPECTION

--- Disk usage ---
Total:     437.93 GB
Used:      360.62 GB
Free:      54.99 GB

FILE: models/net_utils_e5.py
Total lines: 78

--- Lines 2-19 ---
   2: import torch.nn as nn
   3: 
   4: from .net_utils import FeatureFusionModule
   5: 
   6: 
   7: class ReliabilityGatedFeatureRectifyModule(nn.Module):
   8:     """
   9:     E4 prototype:
  10:     Estimate per-pixel RGB/thermal reliability and use it to
  11:     gate cross-modal feature corrections.
  12: 
  13:     This module does not yet implement counterfactual supervision.
  14:     """
  15: 
  16:     def __init__(self, dim, reduction=1, lambda_c=0.5, lambda_s=0.5):
  17:         super().__init__()
  18: 
  19:         # Reuse the original CMX channel and spatial weighting mechanisms.

--- Lines 31-46 ---
  31: 
  32:         hidden = max(dim // 8, 8)
  33:         self.reliability_head = nn.Sequential(
  34:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  35:             

In [2]:

from pathlib import Path
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

checks = {
    "Config": SCR_DIR / "config.py",
    "Builder": SCR_DIR / "models/builder.py",
    "E5 encoder": SCR_DIR / "models/encoders/dual_segformer_e5.py",
    "E5 reliability module": SCR_DIR / "models/net_utils_e5.py",
    "E3 fusion module": SCR_DIR / "models/net_utils_e3.py",
    "E5 checkpoint": SCR_DIR / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth",
}

print("SCR-CMX workspace verification")
print("=" * 65)
for name, path in checks.items():
    print(f"{name:26} {'OK' if path.exists() else 'MISSING'}")

# Find where FRMs and FFMs are initialized in the encoder
encoder_path = checks["E5 encoder"]
if encoder_path.exists():
    lines = encoder_path.read_text(errors="replace").splitlines()
    print("\n--- Encoder initialization and stage fusion ---")
    patterns = re.compile(
        r"self\.FRMs|self\.FFMs|ReliabilityGuidedFusionModule|"
        r"FeatureFusionModule|def __init__|def forward_features"
    )
    matching = [i for i, line in enumerate(lines) if patterns.search(line)]
    shown = set()
    for i in matching:
        for j in range(max(0, i - 3), min(len(lines), i + 5)):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

# Show dataset and backbone configuration
config_path = checks["Config"]
if config_path.exists():
    lines = config_path.read_text(errors="replace").splitlines()
    print("\n--- Relevant configuration ---")
    patterns = re.compile(
        r"C\.backbone|C\.dataset_path|C\.dataset_name|C\.num_classes|"
        r"C\.batch_size|C\.nepochs|C\.lr|C\.niters_per_epoch|"
        r"C\.pretrained_model|C\.log_dir"
    )
    for i, line in enumerate(lines):
        if patterns.search(line):
            print(f"{i + 1:4}: {line}")

print("\nVerification finished; no files changed and no training started.")


SCR-CMX workspace verification
Config                     OK
Builder                    OK
E5 encoder                 OK
E5 reliability module      OK
E3 fusion module           OK
E5 checkpoint              OK

--- Encoder initialization and stage fusion ---
   4: from functools import partial
   5: 
   6: from timm.models.layers import DropPath, to_2tuple, trunc_normal_
   7: from ..net_utils_e3 import ReliabilityGuidedFusionModule as FFM
   8: from ..net_utils_e5 import ReliabilityGatedFeatureRectifyModule as FRM
   9: import math
  10: import time
  11: from engine.logger import get_logger
  17:     """
  18:     Depthwise convolution bloc: input: x with size(B N C); output size (B N C)
  19:     """
  20:     def __init__(self, dim=768):
  21:         super(DWConv, self).__init__()
  22:         self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, bias=True, groups=dim)
  23: 
  24:     def forward(self, x, H, W):
  31: 
  32: 
  33: class Mlp(nn.Module):
  34:   

In [3]:

from pathlib import Path
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_inspect = {
    "E3 fusion module": SCR_DIR / "models/net_utils_e3.py",
    "E5 encoder": SCR_DIR / "models/encoders/dual_segformer_e5.py",
    "E5 builder": SCR_DIR / "models/builder.py",
}

patterns = {
    "E3 fusion module": re.compile(
        r"class ReliabilityGuidedFusionModule|"
        r"def __init__|def forward|self\.|return "
    ),
    "E5 encoder": re.compile(
        r"self\.FRMs\s*=|self\.FFMs\s*=|"
        r"def forward_features|self\.last_reliabilities|"
        r"self\.FRMs\[|self\.FFMs\["
    ),
    "E5 builder": re.compile(
        r"class EncoderDecoder|def __init__|"
        r"def encode_decode|def _e5_counterfactual_loss|"
        r"def forward|mit_b2_e5|last_reliabilities"
    ),
}

for name, path in files_to_inspect.items():
    print("\n" + "=" * 90)
    print(name, "|", "EXISTS:", path.exists())
    print("=" * 90)

    if not path.exists():
        continue

    lines = path.read_text(errors="replace").splitlines()
    selected = set()

    for i, line in enumerate(lines):
        if patterns[name].search(line):
            selected.update(
                range(max(0, i - 2), min(len(lines), i + 4))
            )

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")
        print(f"{i + 1:4}: {lines[i]}")
        previous = i

print("\nInspection complete. No source files modified; no training started.")



E3 fusion module | EXISTS: True
    ...
   5: 
   6: 
   7: class ReliabilityGuidedFusionModule(FeatureFusionModule):
   8:     """
   9:     E3 prototype:
  10:     Preserve CMX CrossPath and ChannelEmbed, but estimate spatially
    ...
  12:     """
  13: 
  14:     def __init__(self, dim, reduction=1, num_heads=None,
  15:                  norm_layer=nn.BatchNorm2d):
  16:         super().__init__(
  17:             dim=dim,
    ...
  22: 
  23:         hidden = max(dim // 8, 8)
  24:         self.reliability_head = nn.Sequential(
  25:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  26:             nn.ReLU(inplace=True),
  27:             nn.Conv2d(hidden, 2, kernel_size=1),
  28:         )
  29: 
  30:     def forward(self, x1, x2):
  31:         B, C, H, W = x1.shape
  32: 
  33:         # Original CMX cross-modal interaction.
  34:         t1 = x1.flatten(2).transpose(1, 2)
  35:         t2 = x2.flatten(2).transpose(1, 2)
  36:         t1, t2 = self.cross(t1, t2)
  37:

In [4]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

targets = [
    ("E3 reliability-guided fusion",
     SCR_DIR / "models/net_utils_e3.py",
     "class ReliabilityGuidedFusionModule"),
    ("Original CMX fusion base",
     SCR_DIR / "models/net_utils.py",
     "class FeatureFusionModule"),
]

for title, path, marker in targets:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    if not path.exists():
        print("FILE MISSING:", path)
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if marker in line
    ]

    if not matches:
        print("Class not found:", marker)
        continue

    for start in matches:
        # Print the complete class until the next top-level class.
        end = len(lines)
        for i in range(start + 1, len(lines)):
            if lines[i].startswith("class "):
                end = i
                break

        print(f"\nLines {start + 1}-{end}:")
        for i in range(start, end):
            print(f"{i + 1:4}: {lines[i]}")

print("\nInspection complete. No files modified.")



E3 reliability-guided fusion

Lines 7-53:
   7: class ReliabilityGuidedFusionModule(FeatureFusionModule):
   8:     """
   9:     E3 prototype:
  10:     Preserve CMX CrossPath and ChannelEmbed, but estimate spatially
  11:     varying RGB/thermal fusion weights after cross-modal interaction.
  12:     """
  13: 
  14:     def __init__(self, dim, reduction=1, num_heads=None,
  15:                  norm_layer=nn.BatchNorm2d):
  16:         super().__init__(
  17:             dim=dim,
  18:             reduction=reduction,
  19:             num_heads=num_heads,
  20:             norm_layer=norm_layer,
  21:         )
  22: 
  23:         hidden = max(dim // 8, 8)
  24:         self.reliability_head = nn.Sequential(
  25:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  26:             nn.ReLU(inplace=True),
  27:             nn.Conv2d(hidden, 2, kernel_size=1),
  28:         )
  29: 
  30:     def forward(self, x1, x2):
  31:         B, C, H, W = x1.shape
  32: 
  33:         # 

In [5]:

from pathlib import Path
import shutil

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

fusion_file = SCR_DIR / "models/net_utils_scr.py"
encoder_file = SCR_DIR / "models/encoders/dual_segformer_e5.py"
backup_file = SCR_DIR / "models/encoders/dual_segformer_e5.py.before_scr"

assert fusion_file.parent.exists(), "SCR models directory not found"
assert encoder_file.exists(), "SCR E5 encoder not found"
assert not fusion_file.exists(), (
    "SCR fusion file already exists. Stop and inspect it before overwriting."
)

# Preserve the original SCR copy of the encoder before editing.
if not backup_file.exists():
    shutil.copy2(encoder_file, backup_file)

fusion_source = r'''
import torch
import torch.nn as nn

from .net_utils import FeatureFusionModule


class SpatialReliabilityConditionedFusion(FeatureFusionModule):
    """
    SCR-CMX: Spatial Reliability-Conditioned Cross-Modal Interaction.

    Reuses the CMX CrossPath and ChannelEmbed modules.
    Reliability controls the cross-modal residual update rather than
    weighting the completed cross-modal features after interaction.
    """

    def __init__(
        self,
        dim,
        reduction=1,
        num_heads=None,
        norm_layer=nn.BatchNorm2d,
    ):
        super().__init__(
            dim=dim,
            reduction=reduction,
            num_heads=num_heads,
            norm_layer=norm_layer,
        )

    def forward(self, x_rgb, x_thermal, reliability):
        if x_rgb.shape != x_thermal.shape:
            raise ValueError(
                f"RGB and thermal feature shapes differ: "
                f"{x_rgb.shape} vs {x_thermal.shape}"
            )

        if reliability.ndim != 4 or reliability.shape[1] != 2:
            raise ValueError(
                "Expected reliability shape (B, 2, H, W), "
                f"received {tuple(reliability.shape)}"
            )

        B, C, H, W = x_rgb.shape

        if reliability.shape[0] != B or reliability.shape[2:] != (H, W):
            raise ValueError(
                "Reliability batch/spatial dimensions must match features: "
                f"{tuple(reliability.shape)} vs {tuple(x_rgb.shape)}"
            )

        r_rgb = reliability[:, 0:1].clamp(0.0, 1.0)
        r_thermal = reliability[:, 1:2].clamp(0.0, 1.0)

        # Original features as token sequences.
        rgb_tokens = x_rgb.flatten(2).transpose(1, 2)
        thermal_tokens = x_thermal.flatten(2).transpose(1, 2)

        # Existing CMX cross-modal interaction.
        rgb_cross, thermal_cross = self.cross(rgb_tokens, thermal_tokens)

        # Return to spatial feature maps.
        rgb_cross = (
            rgb_cross.transpose(1, 2)
            .reshape(B, C, H, W)
            .contiguous()
        )
        thermal_cross = (
            thermal_cross.transpose(1, 2)
            .reshape(B, C, H, W)
            .contiguous()
        )

        # Interaction deltas: isolate the changes introduced by CrossPath.
        rgb_delta = rgb_cross - x_rgb
        thermal_delta = thermal_cross - x_thermal

        # Reliability-conditioned cross-modal updates.
        # RGB receives thermal information according to thermal reliability.
        # Thermal receives RGB information according to RGB reliability.
        rgb_out = x_rgb + r_thermal * rgb_delta
        thermal_out = x_thermal + r_rgb * thermal_delta

        # Keep the original CMX channel embedding interface.
        rgb_tokens = rgb_out.flatten(2).transpose(1, 2)
        thermal_tokens = thermal_out.flatten(2).transpose(1, 2)
        merged = torch.cat([rgb_tokens, thermal_tokens], dim=-1)

        return self.channel_emb(merged, H, W)
'''

fusion_file.write_text(fusion_source.strip() + "\n")

print("Created:", fusion_file)
print("Encoder backup:", backup_file)
print("Original E0-E5 workspaces were not modified.")
print("No training started.")


Created: /home/cse-sdpl/Downloads/CMX_SCR/models/net_utils_scr.py
Encoder backup: /home/cse-sdpl/Downloads/CMX_SCR/models/encoders/dual_segformer_e5.py.before_scr
Original E0-E5 workspaces were not modified.
No training started.


In [6]:

from pathlib import Path
import ast
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files = {
    "SCR fusion": SCR_DIR / "models/net_utils_scr.py",
    "CMX base modules": SCR_DIR / "models/net_utils.py",
    "E5 encoder": SCR_DIR / "models/encoders/dual_segformer_e5.py",
}

for name, path in files.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

# Show CrossPath and ChannelEmbed definitions with line numbers.
base_file = files["CMX base modules"]
lines = base_file.read_text(errors="replace").splitlines()

for class_name in ("class CrossPath", "class ChannelEmbed"):
    starts = [i for i, line in enumerate(lines) if line.startswith(class_name)]
    for start in starts:
        end = len(lines)
        for i in range(start + 1, len(lines)):
            if lines[i].startswith("class "):
                end = i
                break
        print(f"\n--- {class_name} (lines {start+1}-{end}) ---")
        for i in range(start, end):
            print(f"{i+1:4}: {lines[i]}")

# Inspect the encoder's existing reliability and fusion calls.
encoder_file = files["E5 encoder"]
enc_lines = encoder_file.read_text(errors="replace").splitlines()

print("\n--- Encoder FRM/FFM setup and calls ---")
pattern = re.compile(r"self\.FRMs\s*=|self\.FFMs\s*=|self\.FRMs\[|self\.FFMs\[")
for i, line in enumerate(enc_lines):
    if pattern.search(line):
        lo, hi = max(0, i-2), min(len(enc_lines), i+4)
        for j in range(lo, hi):
            print(f"{j+1:4}: {enc_lines[j]}")
        print()

print("\nInspection only: no files modified and no training started.")


SCR fusion: OK
CMX base modules: OK
E5 encoder: OK

--- class CrossPath (lines 111-135) ---
 111: class CrossPath(nn.Module):
 112:     def __init__(self, dim, reduction=1, num_heads=None, norm_layer=nn.LayerNorm):
 113:         super().__init__()
 114:         self.channel_proj1 = nn.Linear(dim, dim // reduction * 2)
 115:         self.channel_proj2 = nn.Linear(dim, dim // reduction * 2)
 116:         self.act1 = nn.ReLU(inplace=True)
 117:         self.act2 = nn.ReLU(inplace=True)
 118:         self.cross_attn = CrossAttention(dim // reduction, num_heads=num_heads)
 119:         self.end_proj1 = nn.Linear(dim // reduction * 2, dim)
 120:         self.end_proj2 = nn.Linear(dim // reduction * 2, dim)
 121:         self.norm1 = norm_layer(dim)
 122:         self.norm2 = norm_layer(dim)
 123: 
 124:     def forward(self, x1, x2):
 125:         y1, u1 = self.act1(self.channel_proj1(x1)).chunk(2, dim=-1)
 126:         y2, u2 = self.act2(self.channel_proj2(x2)).chunk(2, dim=-1)
 127:       

In [7]:

from pathlib import Path
import re
import py_compile

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
encoder_path = SCR_DIR / "models/encoders/dual_segformer_e5.py"
fusion_path = SCR_DIR / "models/net_utils_scr.py"
backup_path = SCR_DIR / "models/encoders/dual_segformer_e5.py.before_scr"

assert encoder_path.exists(), f"Missing encoder: {encoder_path}"
assert fusion_path.exists(), f"Missing SCR fusion module: {fusion_path}"
assert backup_path.exists(), f"Missing original encoder backup: {backup_path}"

source = encoder_path.read_text()

# Avoid accidentally applying the patch twice.
if "SCR-CMX: reliability-conditioned fusion" in source:
    print("SCR encoder patch already present; no changes made.")
else:
    # Import the new SCR fusion module.
    import_line = (
        "from ..net_utils_e5 import ReliabilityGatedFeatureRectifyModule as FRM"
    )
    scr_import = (
        import_line
        + "\nfrom ..net_utils_scr import SpatialReliabilityConditionedFusion as SCRFFM"
    )
    assert import_line in source, "Could not find the expected E5 FRM import."
    source = source.replace(import_line, scr_import, 1)

    # Replace the four fusion modules with SCR fusion modules.
    old_ffm_block = re.compile(
        r"        self\.FFMs = nn\.ModuleList\(\[\s*"
        r"FFM\(dim=embed_dims\[0\].*?\n\s*"
        r"FFM\(dim=embed_dims\[3\].*?\]\)",
        re.DOTALL,
    )

    replacement = '''        # SCR-CMX: reliability-conditioned fusion at all four stages.
        self.FFMs = nn.ModuleList([
            SCRFFM(dim=embed_dims[0], reduction=1, num_heads=num_heads[0], norm_layer=norm_fuse),
            SCRFFM(dim=embed_dims[1], reduction=1, num_heads=num_heads[1], norm_layer=norm_fuse),
            SCRFFM(dim=embed_dims[2], reduction=1, num_heads=num_heads[2], norm_layer=norm_fuse),
            SCRFFM(dim=embed_dims[3], reduction=1, num_heads=num_heads[3], norm_layer=norm_fuse),
        ])'''

    source, count = old_ffm_block.subn(replacement, source, count=1)
    assert count == 1, (
        "Could not safely locate the original four-stage FFM block. "
        "Encoder was not written."
    )

    # Pass each existing E5 reliability map to SCR fusion.
    for stage in range(4):
        old_call = f"x_fused = self.FFMs[{stage}](x_rgb, x_e)"
        new_call = (
            f"# SCR-CMX: reliability-conditioned fusion\n"
            f"        x_fused = self.FFMs[{stage}](x_rgb, x_e, reliability)"
        )
        assert old_call in source, f"Missing expected fusion call for stage {stage + 1}"
        source = source.replace(old_call, new_call, 1)

    # Validate Python syntax before writing the source.
    compile(source, str(encoder_path), "exec")

    encoder_path.write_text(source)
    print("SCR encoder patched successfully.")

# Compile both files without importing or training the model.
py_compile.compile(str(encoder_path), doraise=True)
py_compile.compile(str(fusion_path), doraise=True)

print("Encoder syntax: OK")
print("SCR fusion syntax: OK")
print("Original E5 encoder backup remains:", backup_path)
print("No training started.")


SCR encoder patched successfully.
Encoder syntax: OK
SCR fusion syntax: OK
Original E5 encoder backup remains: /home/cse-sdpl/Downloads/CMX_SCR/models/encoders/dual_segformer_e5.py.before_scr
No training started.


In [8]:

from pathlib import Path
import sys
import torch
import traceback

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = (
    SCR_DIR / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
)

assert SCR_DIR.exists(), f"SCR directory missing: {SCR_DIR}"
assert E5_CHECKPOINT.exists(), f"E5 checkpoint missing: {E5_CHECKPOINT}"

# Run imports from the SCR workspace.
sys.path.insert(0, str(SCR_DIR))
import os
os.chdir(SCR_DIR)

import config
from models.builder import EncoderDecoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Match the E5 configuration but avoid creating a training run.
cfg = config.C
cfg.backbone = "mit_b2_e5"
cfg.num_classes = 9

model = EncoderDecoder(cfg=cfg).to(device)
print("Model constructed.")

# Inspect the checkpoint structure.
checkpoint = torch.load(
    str(E5_CHECKPOINT),
    map_location="cpu",
    weights_only=False,
)

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:20])

    if "model" in checkpoint:
        state_dict = checkpoint["model"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
    else:
        # Some training scripts save the state dictionary directly.
        state_dict = checkpoint
else:
    raise TypeError(f"Unexpected checkpoint type: {type(checkpoint)}")

# Load only shape-compatible keys; report exactly what was skipped.
current_state = model.state_dict()
compatible = {
    key: value
    for key, value in state_dict.items()
    if key in current_state
    and hasattr(value, "shape")
    and current_state[key].shape == value.shape
}

print("Checkpoint tensors:", len(state_dict))
print("Shape-compatible tensors:", len(compatible))
print("Model tensors:", len(current_state))

load_result = model.load_state_dict(compatible, strict=False)
print("Missing model keys:", len(load_result.missing_keys))
print("Unexpected checkpoint keys:", len(load_result.unexpected_keys))
print("Sample missing keys:", load_result.missing_keys[:15])

# Synthetic forward/backward test at 640 x 480.
model.train()
model.zero_grad(set_to_none=True)

rgb = torch.randn(1, 3, 480, 640, device=device)
thermal = torch.randn(1, 3, 480, 640, device=device)
labels = torch.randint(0, 9, (1, 480, 640), device=device)

try:
    with torch.enable_grad():
        loss = model(rgb, thermal, labels)

        if isinstance(loss, (tuple, list)):
            loss = sum(x for x in loss if torch.is_tensor(x) and x.numel() == 1)

        if not torch.is_tensor(loss) or loss.numel() != 1:
            raise RuntimeError(
                f"Expected scalar training loss, got {type(loss)} "
                f"with shape {getattr(loss, 'shape', None)}"
            )

        print("Loss:", float(loss.detach().cpu()))
        print("Loss finite:", bool(torch.isfinite(loss).all()))

        loss.backward()

    grads = [
        p.grad for p in model.parameters()
        if p.requires_grad and p.grad is not None
    ]
    finite_grads = all(bool(torch.isfinite(g).all()) for g in grads)

    print("Parameters with gradients:", len(grads))
    print("All observed gradients finite:", finite_grads)

    scr_parameters = [
        (name, p) for name, p in model.named_parameters()
        if "FFMs" in name and p.requires_grad
    ]
    scr_grad_count = sum(p.grad is not None for _, p in scr_parameters)

    print("Trainable SCR fusion parameters:", len(scr_parameters))
    print("SCR fusion parameters receiving gradients:", scr_grad_count)

    if not bool(torch.isfinite(loss).all()) or not finite_grads:
        raise RuntimeError("Non-finite loss or gradients detected.")

    print("\nSMOKE TEST PASSED")

except Exception:
    print("\nSMOKE TEST FAILED")
    traceback.print_exc()
    raise

finally:
    del rgb, thermal, labels
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("No optimizer step was performed; no training checkpoint was saved.")


23 17:56:42 Using E5: SegFormer-B2 with counterfactual reliability


Device: cuda


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
23 17:56:44 Using MLP Decoder
23 17:56:44 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 17:56:44 Load model, Time usage:
	IO: 0.05199480056762695, initialize parameters: 0.027271032333374023
23 17:56:44 Initing weights ...


Model constructed.
Checkpoint keys: ['model', 'optimizer', 'epoch', 'iteration']
Checkpoint tensors: 869
Shape-compatible tensors: 853
Model tensors: 853
Missing model keys: 0
Unexpected checkpoint keys: 0
Sample missing keys: []
Loss: 18.851318359375
Loss finite: True
Parameters with gradients: 826
All observed gradients finite: True
Trainable SCR fusion parameters: 100
SCR fusion parameters receiving gradients: 100

SMOKE TEST PASSED
No optimizer step was performed; no training checkpoint was saved.


In [9]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_check = [
    SCR_DIR / "train.py",
    SCR_DIR / "models/builder.py",
    SCR_DIR / "models/encoders/dual_segformer_e5.py",
    SCR_DIR / "models/net_utils_scr.py",
]

keywords = [
    "load_state_dict",
    "optimizer",
    "params_list",
    "counterfactual",
    "last_reliabilities",
    "loss.backward",
    "optimizer.step",
    "torch.save",
]

for path in files_to_check:
    print("\n" + "=" * 90)
    print("FILE:", path)
    print("=" * 90)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if any(key.lower() in line.lower() for key in keywords)
    ]

    shown = set()
    for i in matches:
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        if any(j in shown for j in range(start, end)):
            continue

        print(f"\n--- lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")
            shown.add(j)



FILE: /home/cse-sdpl/Downloads/CMX_SCR/train.py

--- lines 59-66 ---
  59:     model=segmodel(cfg=config, criterion=criterion, norm_layer=BatchNorm2d)
  60: 
  61:     # E3: initialize compatible model weights from E2 only.
  62:     # Do not restore E2 optimizer, epoch, or training state.
  63:     if config.backbone == 'mit_b2_e3':
  64:         import torch
  65: 
  66:         e2_checkpoint_path = r"/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth"

--- lines 77-84 ---
  77:             if key in e3_state and e3_state[key].shape == value.shape
  78:         }
  79: 
  80:         result = model.load_state_dict(compatible_state, strict=False)
  81: 
  82:         # Only E3 reliability-head parameters should be missing.
  83:         assert not result.unexpected_keys, result.unexpected_keys
  84:         assert all(

--- lines 100-107 ---
 100:     
 101: 
 102:     # E4: initialize compatible model weights from E3 only.
 103:     # Do not restore the E3 op

In [10]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_inspect = {
    "config.py": SCR_DIR / "config.py",
    "builder.py": SCR_DIR / "models" / "builder.py",
    "train.py": SCR_DIR / "train.py",
}

keywords = [
    "backbone",
    "log_path",
    "log_dir",
    "checkpoint",
    "snapshot",
    "save",
    "continue_state_object",
    "mit_b2_e5",
    "e5_counterfactual",
    "e5_reliability_loss_weight",
    "e5_corruption_fraction",
    "e5_counterfactual_temperature",
]

for label, path in files_to_inspect.items():
    print("\n" + "=" * 90)
    print(label, "| exists:", path.exists())
    print("=" * 90)

    if not path.exists():
        continue

    lines = path.read_text(errors="replace").splitlines()
    matched = set()

    for i, line in enumerate(lines):
        if any(key.lower() in line.lower() for key in keywords):
            start = max(0, i - 2)
            end = min(len(lines), i + 4)

            if any(j in matched for j in range(start, end)):
                continue

            print(f"\n--- lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")
                matched.add(j)



config.py | exists: True

--- lines 79-84 ---
  79: # ============================================================
  80: # Network Configuration
  81: C.backbone = 'mit_b2_e5'
  82: 
  83: C.pretrained_model = '/home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth'
  84: 

--- lines 118-123 ---
 118: 
 119: # ============================================================
 120: # CHECKPOINT CONFIGURATION
 121: # ============================================================
 122: C.checkpoint_start_epoch = 1
 123: C.checkpoint_step = 5

--- lines 135-140 ---
 135: # LOGGING CONFIGURATION
 136: # ============================================================
 137: C.log_dir = osp.abspath(
 138:     'log_' + C.dataset_name + '_' + C.backbone
 139: )
 140: 

--- lines 141-146 ---
 141: C.tb_dir = osp.join(C.log_dir, 'tb')
 142: C.log_dir_link = C.log_dir
 143: C.checkpoint_dir = osp.join(C.log_dir, 'checkpoint')
 144: 
 145: exp_time = time.strftime('%Y_%m_%d_%H_%M_%S', time.loc

In [11]:

from pathlib import Path
import shutil
import py_compile

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

config_path = SCR_DIR / "config.py"
builder_path = SCR_DIR / "models" / "builder.py"
train_path = SCR_DIR / "train.py"

# Read and validate before making any changes.
config_text = config_path.read_text()
builder_text = builder_path.read_text()
train_text = train_path.read_text()

assert "C.backbone = 'mit_b2_e5'" in config_text, \
    "Unexpected config.py backbone line; no changes made."

assert "elif cfg.backbone == 'mit_b2_e5':" in builder_text, \
    "Expected E5 builder branch not found; no changes made."

assert 'if config.backbone == \'mit_b2_e5\':' in train_text, \
    "Expected E5 initialization branch not found; no changes made."

# Back up the three files once.
for path in (config_path, builder_path, train_path):
    backup = path.with_name(path.name + ".before_scr_identity")
    if not backup.exists():
        shutil.copy2(path, backup)
    print("Backup:", backup)

# 1. Give SCR its own backbone name.
config_text = config_text.replace(
    "C.backbone = 'mit_b2_e5'",
    "C.backbone = 'mit_b2_scr'",
    1,
)

# 2. Add SCR builder branch before the E5 branch.
scr_branch = """        elif cfg.backbone == 'mit_b2_scr':
            logger.info('Using SCR-CMX: Spatial Reliability-Conditioned Fusion')
            from .encoders.dual_segformer_e5 import mit_b2 as backbone
            self.backbone = backbone(norm_fuse=norm_layer)
"""

builder_text = builder_text.replace(
    "        elif cfg.backbone == 'mit_b2_e5':",
    scr_branch + "        elif cfg.backbone == 'mit_b2_e5':",
    1,
)

# 3. Allow the counterfactual reliability loss to be explicitly enabled
# for SCR without changing E5's existing behavior.
old_flag = '''        self.e5_counterfactual = (
            getattr(cfg, "backbone", "") == "mit_b2_e5"
        )'''

new_flag = '''        self.e5_counterfactual = (
            getattr(cfg, "backbone", "") == "mit_b2_e5"
            or (
                getattr(cfg, "backbone", "") == "mit_b2_scr"
                and getattr(cfg, "scr_use_counterfactual_loss", True)
            )
        )'''

assert old_flag in builder_text, \
    "Counterfactual flag pattern not found; no changes written."

builder_text = builder_text.replace(old_flag, new_flag, 1)

# 4. Add a SCR-specific E5 initialization branch in train.py.
# It loads model weights only; the optimizer and schedule remain fresh.
scr_init = '''    # SCR: initialize compatible model weights from trained E5.
    # Do not restore E5 optimizer, epoch, or training state.
    if config.backbone == 'mit_b2_scr':
        import torch

        scr_init_path = (
            "/home/cse-sdpl/Downloads/CMX_SCR/"
            "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
        )
        if not __import__("os").path.isfile(scr_init_path):
            raise FileNotFoundError(
                f"SCR initialization checkpoint not found: {scr_init_path}"
            )

        scr_checkpoint = torch.load(
            scr_init_path,
            map_location="cpu",
            weights_only=False,
        )
        scr_state = (
            scr_checkpoint["model"]
            if isinstance(scr_checkpoint, dict)
            and "model" in scr_checkpoint
            else scr_checkpoint
        )
        current_state = model.state_dict()
        compatible_state = {
            key: value
            for key, value in scr_state.items()
            if key in current_state
            and current_state[key].shape == value.shape
        }

        result = model.load_state_dict(compatible_state, strict=False)
        print("SCR initialized from E5:", scr_init_path)
        print("Compatible checkpoint tensors:", len(compatible_state))
        print("Missing model keys:", len(result.missing_keys))
        print("Unexpected checkpoint keys:", len(result.unexpected_keys))
        print("Sample missing keys:", result.missing_keys[:20])

        assert not result.unexpected_keys, result.unexpected_keys

'''

train_text = train_text.replace(
    "    # E5: initialize model weights from the trained E4 checkpoint.",
    scr_init + "    # E5: initialize model weights from the trained E4 checkpoint.",
    1,
)

# 5. Write only after all expected patterns have been validated.
config_path.write_text(config_text)
builder_path.write_text(builder_text)
train_path.write_text(train_text)

# Syntax check all modified Python files.
for path in (config_path, builder_path, train_path):
    py_compile.compile(str(path), doraise=True)
    print("Syntax OK:", path.name)

print("\nSCR identity patch applied.")
print("Backbone:", "mit_b2_scr")
print("Counterfactual reliability loss: enabled by default for SCR")
print("Training started: NO")


Backup: /home/cse-sdpl/Downloads/CMX_SCR/config.py.before_scr_identity
Backup: /home/cse-sdpl/Downloads/CMX_SCR/models/builder.py.before_scr_identity
Backup: /home/cse-sdpl/Downloads/CMX_SCR/train.py.before_scr_identity
Syntax OK: config.py
Syntax OK: builder.py
Syntax OK: train.py

SCR identity patch applied.
Backbone: mit_b2_scr
Counterfactual reliability loss: enabled by default for SCR
Training started: NO


In [12]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

config_text = (SCR_DIR / "config.py").read_text()
builder_text = (SCR_DIR / "models" / "builder.py").read_text()
train_text = (SCR_DIR / "train.py").read_text()

checks = {
    "SCR backbone selected in config":
        "C.backbone = 'mit_b2_scr'" in config_text,

    "Separate SCR builder branch":
        "elif cfg.backbone == 'mit_b2_scr':" in builder_text,

    "SCR-specific checkpoint initialization":
        "if config.backbone == 'mit_b2_scr':" in train_text,

    "E5 branch retained":
        "if config.backbone == 'mit_b2_e5':" in train_text,

    "SCR-specific log directory will be derived from backbone":
        "C.log_dir = osp.abspath(" in config_text
        and "C.backbone" in config_text,

    "SCR initialization checkpoint path present":
        "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth" in train_text,
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

assert all(checks.values()), "One or more SCR patch checks failed."

print("\nVerification passed. No training has been started.")
print("Expected SCR log directory: log_TDD_mit_b2_scr")


PASS - SCR backbone selected in config
PASS - Separate SCR builder branch
PASS - SCR-specific checkpoint initialization
PASS - E5 branch retained
PASS - SCR-specific log directory will be derived from backbone
PASS - SCR initialization checkpoint path present

Verification passed. No training has been started.
Expected SCR log directory: log_TDD_mit_b2_scr


In [13]:

from pathlib import Path
import py_compile
import shutil

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
train_path = SCR_DIR / "train.py"
text = train_path.read_text()

old = "if config.backbone in ('mit_b2_e3', 'mit_b2_e4'):"
new = "if config.backbone in ('mit_b2_e3', 'mit_b2_e4', 'mit_b2_scr'):"

if new in text:
    print("SCR optimizer audit is already enabled.")
elif old in text:
    backup = train_path.with_name("train.py.before_scr_optimizer_audit")
    if not backup.exists():
        shutil.copy2(train_path, backup)
    text = text.replace(old, new, 1)
    train_path.write_text(text)
    py_compile.compile(str(train_path), doraise=True)
    print("SCR optimizer audit added successfully.")
    print("Backup:", backup)
else:
    raise RuntimeError(
        "Expected optimizer audit condition not found. No changes made."
    )

# Verify the final condition.
text = train_path.read_text()
assert new in text
print("Verified: SCR is included in the optimizer audit.")
print("Training started: NO")


SCR optimizer audit added successfully.
Backup: /home/cse-sdpl/Downloads/CMX_SCR/train.py.before_scr_optimizer_audit
Verified: SCR is included in the optimizer audit.
Training started: NO


In [14]:

from pathlib import Path
import os
import sys

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = (
    SCR_DIR / "log_TDD_mit_b2_e5" / "checkpoint" / "epoch-100.pth"
)

os.chdir(SCR_DIR)
if str(SCR_DIR) not in sys.path:
    sys.path.insert(0, str(SCR_DIR))

import config

cfg = config.C

print("Backbone:", cfg.backbone)
print("Log directory:", cfg.log_dir)
print("Checkpoint directory:", cfg.checkpoint_dir)
print("Dataset path:", cfg.dataset_path)
print("E5 initialization checkpoint exists:", E5_CHECKPOINT.is_file())
print("SCR counterfactual loss setting:",
      getattr(cfg, "scr_use_counterfactual_loss", True))
print("Counterfactual loss weight:",
      getattr(cfg, "e5_reliability_loss_weight", 0.1))

assert cfg.backbone == "mit_b2_scr"
assert Path(cfg.log_dir).name == "log_TDD_mit_b2_scr"
assert Path(cfg.checkpoint_dir).parent == Path(cfg.log_dir)
assert E5_CHECKPOINT.is_file(), (
    f"Missing initialization checkpoint: {E5_CHECKPOINT}"
)

print("\nPREFLIGHT PASSED")
print("No training started and no checkpoint saved.")


Backbone: mit_b2_e5
Log directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5
Checkpoint directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint
Dataset path: /home/cse-sdpl/Downloads/CMX_Original/datasets/TDD
E5 initialization checkpoint exists: True
SCR counterfactual loss setting: True
Counterfactual loss weight: 0.1


AssertionError: 

In [15]:

from pathlib import Path
import os
import sys
import importlib

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
os.chdir(SCR_DIR)

if str(SCR_DIR) not in sys.path:
    sys.path.insert(0, str(SCR_DIR))

# Remove the cached module before importing the SCR configuration.
if "config" in sys.modules:
    del sys.modules["config"]

import config
importlib.reload(config)

cfg = config.C

print("Loaded config file:", Path(config.__file__).resolve())
print("Backbone:", cfg.backbone)
print("Log directory:", cfg.log_dir)
print("Checkpoint directory:", cfg.checkpoint_dir)

print("\nConfig source lines:")
for i, line in enumerate(
    (SCR_DIR / "config.py").read_text().splitlines(), start=1
):
    if "C.backbone" in line or "C.log_dir =" in line:
        print(f"{i}: {line}")


Loaded config file: /home/cse-sdpl/Downloads/CMX_SCR/config.py
Backbone: mit_b2_scr
Log directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr
Checkpoint directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint

Config source lines:
81: C.backbone = 'mit_b2_scr'
137: C.log_dir = osp.abspath(
138:     'log_' + C.dataset_name + '_' + C.backbone
163:     print('Backbone:', C.backbone)


In [16]:

from pathlib import Path
import os
import sys
import torch

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = (
    SCR_DIR / "log_TDD_mit_b2_e5" / "checkpoint" / "epoch-100.pth"
)

os.chdir(SCR_DIR)
if str(SCR_DIR) not in sys.path:
    sys.path.insert(0, str(SCR_DIR))

import config
from models.builder import EncoderDecoder

cfg = config.C

assert cfg.backbone == "mit_b2_scr", cfg.backbone
assert E5_CHECKPOINT.is_file(), f"Missing checkpoint: {E5_CHECKPOINT}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Backbone:", cfg.backbone)
print("SCR output directory:", cfg.log_dir)

# Construct the SCR model. No training or checkpoint saving.
model = EncoderDecoder(cfg=cfg).to(device)
print("SCR model constructed.")

# Load compatible E5 weights only; do not restore optimizer or epoch.
checkpoint = torch.load(
    E5_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)
if isinstance(checkpoint, dict) and "model" in checkpoint:
    source_state = checkpoint["model"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    source_state = checkpoint["state_dict"]
else:
    source_state = checkpoint

target_state = model.state_dict()
compatible = {
    key: value
    for key, value in source_state.items()
    if key in target_state and target_state[key].shape == value.shape
}

result = model.load_state_dict(compatible, strict=False)

print("\nCheckpoint audit")
print("Source tensors:", len(source_state))
print("Compatible tensors loaded:", len(compatible))
print("Target model tensors:", len(target_state))
print("Missing target keys:", len(result.missing_keys))
print("Unexpected keys:", len(result.unexpected_keys))
print("Sample missing keys:", result.missing_keys[:15])

# Verify optimizer coverage without performing an optimizer step.
trainable = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]
optimizer = torch.optim.AdamW(trainable, lr=cfg.lr)

model_parameter_ids = {id(p) for p in trainable}
optimizer_parameter_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

missing_from_optimizer = model_parameter_ids - optimizer_parameter_ids
duplicate_count = sum(
    len(group["params"]) for group in optimizer.param_groups
) - len(optimizer_parameter_ids)

print("\nOptimizer audit")
print("Trainable parameter tensors:", len(trainable))
print("Optimizer parameter tensors:", len(optimizer_parameter_ids))
print("Missing from optimizer:", len(missing_from_optimizer))
print("Duplicate optimizer entries:", duplicate_count)

assert len(compatible) > 0, "No checkpoint weights matched."
assert not result.unexpected_keys, result.unexpected_keys
assert not missing_from_optimizer, "Trainable parameters missing from optimizer."
assert duplicate_count == 0, "Duplicate optimizer entries detected."

print("\nSCR MODEL AND BASIC OPTIMIZER AUDIT PASSED")
print("No optimizer step performed; no checkpoint saved.")

del optimizer, model, checkpoint, source_state, target_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()


23 18:05:28 Using backbone: Segformer-B2


Device: cuda
Backbone: mit_b2_scr
SCR output directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr


23 18:05:29 Using MLP Decoder
23 18:05:29 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 18:05:29 Load model, Time usage:
	IO: 0.0792698860168457, initialize parameters: 0.03256511688232422
23 18:05:29 Initing weights ...


SCR model constructed.

Checkpoint audit
Source tensors: 869
Compatible tensors loaded: 837
Target model tensors: 837
Missing target keys: 0
Unexpected keys: 0
Sample missing keys: []

Optimizer audit
Trainable parameter tensors: 810
Optimizer parameter tensors: 810
Missing from optimizer: 0
Duplicate optimizer entries: 0

SCR MODEL AND BASIC OPTIMIZER AUDIT PASSED
No optimizer step performed; no checkpoint saved.


In [17]:

from pathlib import Path
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TRAIN_FILE = ROOT / "train.py"
CONFIG_FILE = ROOT / "config.py"

E5_CHECKPOINT = (
    ROOT / "log_TDD_mit_b2_e5" / "checkpoint" / "epoch-100.pth"
)
SCR_CHECKPOINT_DIR = ROOT / "log_TDD_mit_b2_scr" / "checkpoint"

assert TRAIN_FILE.is_file(), "train.py not found"
assert CONFIG_FILE.is_file(), "config.py not found"
assert E5_CHECKPOINT.is_file(), "SCR initialization checkpoint missing"

train_text = TRAIN_FILE.read_text()
config_text = CONFIG_FILE.read_text()

print("SCR backbone configured:", "C.backbone = 'mit_b2_scr'" in config_text)
print("SCR training initialization reference:", "mit_b2_scr" in train_text)
print("E5 initialization checkpoint exists:", E5_CHECKPOINT.exists())
print("SCR checkpoint directory:", SCR_CHECKPOINT_DIR)
print("SCR output directory differs from E5:", 
      "log_TDD_mit_b2_scr" != "log_TDD_mit_b2_e5")

print("\nRelevant training-script lines:")
for line_no, line in enumerate(train_text.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "mit_b2_scr", "epoch-100.pth", "load_state_dict",
        "optimizer", "checkpoint_dir"
    ]):
        print(f"{line_no}: {line.strip()}")

print("\nNo training started and no checkpoint was modified.")


SCR backbone configured: True
SCR training initialization reference: True
E5 initialization checkpoint exists: True
SCR checkpoint directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint
SCR output directory differs from E5: True

Relevant training-script lines:
62: # Do not restore E2 optimizer, epoch, or training state.
66: e2_checkpoint_path = r"/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth"
80: result = model.load_state_dict(compatible_state, strict=False)
103: # Do not restore the E3 optimizer, epoch, or training state.
109: "log_TDD_mit_b2_e3/checkpoint/epoch-100.pth"
126: result = model.load_state_dict(
162: # Do not restore E5 optimizer, epoch, or training state.
163: if config.backbone == 'mit_b2_scr':
168: "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
194: result = model.load_state_dict(compatible_state, strict=False)
204: # Start a fresh E5 optimizer and training schedule.
208: "log_TDD_mit_b2_e4/checkpoint/epoch-100.pth"
224: re

In [18]:

from pathlib import Path

root = Path("/home/cse-sdpl/Downloads/CMX_SCR")
train_file = root / "train.py"

lines = train_file.read_text().splitlines()

# Display initialization and optimizer construction logic.
start_line = 45
end_line = 310

for i in range(start_line - 1, min(end_line, len(lines))):
    print(f"{i + 1:03d}: {lines[i]}")


045:     if (engine.distributed and (engine.local_rank == 0)) or (not engine.distributed):
046:         tb_dir = config.tb_dir + '/{}'.format(time.strftime("%b%d_%d-%H-%M", time.localtime()))
047:         generate_tb_dir = config.tb_dir + '/tb'
048:         tb = SummaryWriter(log_dir=tb_dir)
049:         engine.link_tb(tb_dir, generate_tb_dir)
050: 
051:     # config network and criterion
052:     criterion = nn.CrossEntropyLoss(reduction='mean', ignore_index=config.background)
053: 
054:     if engine.distributed:
055:         BatchNorm2d = nn.SyncBatchNorm
056:     else:
057:         BatchNorm2d = nn.BatchNorm2d
058:     
059:     model=segmodel(cfg=config, criterion=criterion, norm_layer=BatchNorm2d)
060: 
061:     # E3: initialize compatible model weights from E2 only.
062:     # Do not restore E2 optimizer, epoch, or training state.
063:     if config.backbone == 'mit_b2_e3':
064:         import torch
065: 
066:         e2_checkpoint_path = r"/home/cse-sdpl/Downloads/CMX_E2/log_TD

In [19]:

from pathlib import Path
import py_compile
import shutil

root = Path("/home/cse-sdpl/Downloads/CMX_SCR")
train_file = root / "train.py"

source = train_file.read_text()

old = '''        assert not result.unexpected_keys, result.unexpected_keys

    # E5: initialize model weights from the trained E4 checkpoint.'''

new = '''        assert not result.unexpected_keys, result.unexpected_keys
        assert not result.missing_keys, (
            "SCR checkpoint initialization left model keys missing: "
            f"{result.missing_keys}"
        )
        print("SCR checkpoint compatibility: PASS (no missing or unexpected keys)")

    # E5: initialize model weights from the trained E4 checkpoint.'''

if old not in source:
    raise RuntimeError(
        "Expected SCR initialization block was not found exactly. "
        "No file changes were made."
    )

backup = root / "train.py.before_scr_strict_checkpoint_audit"
if not backup.exists():
    shutil.copy2(train_file, backup)

train_file.write_text(source.replace(old, new, 1))
py_compile.compile(str(train_file), doraise=True)

print("Backup:", backup)
print("SCR strict checkpoint compatibility assertion added.")
print("Syntax check: PASS")
print("Training started: NO")


Backup: /home/cse-sdpl/Downloads/CMX_SCR/train.py.before_scr_strict_checkpoint_audit
SCR strict checkpoint compatibility assertion added.
Syntax check: PASS
Training started: NO


In [20]:

from pathlib import Path
import os
import sys
import torch

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
SCR_OUTPUT = ROOT / "log_TDD_mit_b2_scr"
TRAIN_FILE = ROOT / "train.py"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload configuration so notebook-cached settings cannot interfere.
import importlib
import config
importlib.reload(config)

cfg = config.C

assert cfg.backbone == "mit_b2_scr", cfg.backbone
assert E5_CHECKPOINT.is_file(), f"Missing E5 checkpoint: {E5_CHECKPOINT}"
assert TRAIN_FILE.is_file(), f"Missing training script: {TRAIN_FILE}"
assert "assert not result.missing_keys" in TRAIN_FILE.read_text()

print("Backbone:", cfg.backbone)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("E5 initialization checkpoint:", E5_CHECKPOINT)
print("SCR output directory:", cfg.log_dir)
print("Configured epochs:", cfg.nepochs)
print("Configured batch size:", cfg.batch_size)
print("Configured learning rate:", cfg.lr)
print("E5 checkpoint size (GB):", round(E5_CHECKPOINT.stat().st_size / 1e9, 3))

assert Path(cfg.log_dir).resolve() == SCR_OUTPUT.resolve(), (
    f"Unexpected output directory: {cfg.log_dir}"
)

print("\nPREFLIGHT CONFIGURATION CHECKS PASSED")
print("No training started; no checkpoint saved.")


Backbone: mit_b2_scr
Device: cuda
E5 initialization checkpoint: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
SCR output directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr
Configured epochs: 100
Configured batch size: 2
Configured learning rate: 6e-05
E5 checkpoint size (GB): 0.802

PREFLIGHT CONFIGURATION CHECKS PASSED
No training started; no checkpoint saved.


In [21]:

from pathlib import Path
import os
import sys
import importlib
import torch

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Use the current SCR configuration.
import config
importlib.reload(config)
cfg = config.C

# Clear cached project modules so the current SCR code is imported.
for name in list(sys.modules):
    if name == "models" or name.startswith("models."):
        del sys.modules[name]

from models.builder import EncoderDecoder

assert cfg.backbone == "mit_b2_scr"
assert E5_CHECKPOINT.is_file()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EncoderDecoder(cfg=cfg).to(device)

checkpoint = torch.load(
    E5_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)
source_state = (
    checkpoint["model"]
    if isinstance(checkpoint, dict) and "model" in checkpoint
    else checkpoint
)

target_state = model.state_dict()
compatible_state = {
    key: value
    for key, value in source_state.items()
    if key in target_state and target_state[key].shape == value.shape
}

result = model.load_state_dict(compatible_state, strict=False)

print("Compatible checkpoint tensors:", len(compatible_state))
print("Target model tensors:", len(target_state))
print("Missing keys:", len(result.missing_keys))
print("Unexpected keys:", len(result.unexpected_keys))

assert not result.missing_keys, result.missing_keys
assert not result.unexpected_keys, result.unexpected_keys

scr_params = [
    (name, param)
    for name, param in model.named_parameters()
    if "FFMs" in name and param.requires_grad
]

print("Trainable fusion parameter tensors:", len(scr_params))
assert scr_params, "No trainable fusion parameters found"

print("\nSCR RUNTIME INITIALIZATION: PASS")
print("No optimizer step performed; no checkpoint saved.")

del model, checkpoint, source_state, target_state, compatible_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()


23 18:08:27 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
23 18:08:28 Using MLP Decoder
23 18:08:28 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 18:08:28 Load model, Time usage:
	IO: 0.03600931167602539, initialize parameters: 0.02924036979675293
23 18:08:28 Initing weights ...


Compatible checkpoint tensors: 853
Target model tensors: 853
Missing keys: 0
Unexpected keys: 0
Trainable fusion parameter tensors: 100

SCR RUNTIME INITIALIZATION: PASS
No optimizer step performed; no checkpoint saved.


In [22]:

from pathlib import Path
import os
import subprocess
import sys

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
LOG_DIR = ROOT / "log_TDD_mit_b2_scr"
CHECKPOINT_DIR = LOG_DIR / "checkpoint"

assert (ROOT / "train.py").is_file()
assert (ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth").is_file()

# Protect an existing SCR experiment from accidental overwrite.
if CHECKPOINT_DIR.exists() and any(CHECKPOINT_DIR.iterdir()):
    raise RuntimeError(
        f"SCR checkpoint directory is not empty: {CHECKPOINT_DIR}\n"
        "Inspect or back up the existing run before proceeding."
    )

os.chdir(ROOT)

print("Launching SCR-CMX training...")
print("Workspace:", ROOT)
print("Expected output:", CHECKPOINT_DIR)

result = subprocess.run(
    [
        "conda", "run", "--no-capture-output",
        "-n", "cmx",
        "python", "train.py", "-d", "0"
    ],
    cwd=str(ROOT),
    check=False
)

print("Training process exit code:", result.returncode)


Launching SCR-CMX training...
Workspace: /home/cse-sdpl/Downloads/CMX_SCR
Expected output: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint


23 18:09:14 PyTorch Version 2.14.1+cu130
23 18:09:14 using devices 0
23 18:09:15 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
23 18:09:17 Using MLP Decoder
23 18:09:17 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 18:09:17 Load model, Time usage:
	IO: 0.06279850006103516, initialize parameters: 0.02893972396850586
23 18:09:17 Initing weights ...
23 18:09:18 begin trainning:


SCR initialized from E5: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
Compatible checkpoint tensors: 853
Missing model keys: 0
Unexpected checkpoint keys: 0
Sample missing keys: []
SCR checkpoint compatibility: PASS (no missing or unexpected keys)
Trainable tensors: 826
Optimizer tensors: 826
Reliability-head tensors: 16
Parameters missing from optimizer: []
Duplicate optimizer entries: 0
mit_b2_scr optimizer audit: PASS
[00:00<?,?it/s]

/home/cse-sdpl/Downloads/CMX_SCR/train.py:380: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  + ' loss=%.4f total_loss=%.4f' % (loss, (sum_loss / (idx + 1)))


Epoch 1/100 Iter 785/785: lr=5.9924e-06 loss=0.0766 total_loss=0.0779: [10:52<00:00, 1.20it/s]
Epoch 2/100 Iter 785/785: lr=1.1992e-05 loss=0.0802 total_loss=0.0768: [10:44<00:00, 1.22it/s]
Epoch 3/100 Iter 785/785: lr=1.7992e-05 loss=0.0762 total_loss=0.0771: [10:40<00:00, 1.23it/s]
Epoch 4/100 Iter 785/785: lr=2.3992e-05 loss=0.0750 total_loss=0.0767: [10:39<00:00, 1.23it/s]
Epoch 5/100 Iter 785/785: lr=2.9992e-05 loss=0.0724 total_loss=0.0771: [10:39<00:00, 1.23it/s]

...


Epoch 91/100 Iter 785/785: lr=6.8711e-06 loss=0.0795 total_loss=0.0755: [10:38<00:00, 1.23it/s]
Epoch 92/100 Iter 785/785: lr=6.1801e-06 loss=0.0727 total_loss=0.0756: [10:38<00:00, 1.23it/s]
Epoch 93/100 Iter 785/785: lr=5.4804e-06 loss=0.0751 total_loss=0.0756: [10:38<00:00, 1.23it/s]
Epoch 94/100 Iter 785/785: lr=4.7706e-06 loss=0.0710 total_loss=0.0756: [10:38<00:00, 1.23it/s]
Epoch 95/100 Iter 785/785: lr=4.0488e-06 loss=0.0777 total_loss=0.0754: [10:37<00:00, 1.23it/s]

24 11:01:06 Saving checkpoint to file /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-95.pth
24 11:01:06 Save checkpoint to file /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-95.pth, Time usage:
	prepare checkpoint: 0.00318145751953125, IO: 0.47432470321655273



Epoch 96/100 Iter 785/785: lr=3.3123e-06 loss=0.0729 total_loss=0.0753: [10:38<00:00, 1.23it/s]
Epoch 97/100 Iter 785/785: lr=2.5570e-06 loss=0.0728 total_loss=0.0755: [10:38<00:00, 1.23it/s]
Epoch 98/100 Iter 785/785: lr=1.7755e-06 loss=0.0709 total_loss=0.0753: [10:38<00:00, 1.23it/s]
Epoch 99/100 Iter 785/785: lr=9.5203e-07 loss=0.0781 total_loss=0.0752: [10:38<00:00, 1.23it/s]
Epoch 100/100 Iter 785/785: lr=2.3592e-09 loss=0.0740 total_loss=0.0756: [10:38<00:00, 1.23it/s]

24 11:54:18 Saving checkpoint to file /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth
24 11:54:19 Save checkpoint to file /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth, Time usage:
	prepare checkpoint: 0.0031371116638183594, IO: 0.4719550609588623



Training process exit code: 0


In [23]:

from pathlib import Path

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
CHECKPOINT = ROOT / "log_TDD_mit_b2_scr/checkpoint/epoch-100.pth"

assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"

print("SCR training: COMPLETE")
print("Checkpoint:", CHECKPOINT)
print(f"Checkpoint size: {CHECKPOINT.stat().st_size / (1024**3):.2f} GiB")
print("Ready for evaluation.")


SCR training: COMPLETE
Checkpoint: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth
Checkpoint size: 0.75 GiB
Ready for evaluation.


In [24]:

from pathlib import Path
import subprocess
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
PREDICTIONS = ROOT / "predictions_SCR_test"

assert (ROOT / "eval.py").is_file(), "eval.py not found"
assert (
    ROOT / "log_TDD_mit_b2_scr/checkpoint/epoch-100.pth"
).is_file(), "Final SCR checkpoint not found"

if PREDICTIONS.exists() and any(PREDICTIONS.iterdir()):
    raise RuntimeError(
        f"Prediction directory is not empty: {PREDICTIONS}\n"
        "Choose a fresh output directory to avoid mixing predictions."
    )

os.chdir(ROOT)

command = [
    "conda", "run", "--no-capture-output", "-n", "cmx",
    "python", "eval.py",
    "-e", "100",
    "-d", "0",
    "--split", "test",
    "-p", "predictions_SCR_test",
]

print("Evaluating SCR epoch 100 on the TDD test split...")
print("Command:", " ".join(command))

result = subprocess.run(command, cwd=str(ROOT), check=False)

print("\nEvaluation exit code:", result.returncode)
if result.returncode == 0:
    print("Evaluation process completed. Capture the reported metrics.")
else:
    print("Evaluation failed. Share the error output before retrying.")


Evaluating SCR epoch 100 on the TDD test split...
Command: conda run --no-capture-output -n cmx python eval.py -e 100 -d 0 --split test -p predictions_SCR_test


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 14:34:37 using devices 0
24 14:34:37 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
24 14:34:38 Using MLP Decoder
24 14:34:39 Load Model: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth
24 14:34:39 Load model, Time usage:
	IO: 0.679894208908081, initialize parameters: 0.04182147979736328
24 14:34:39 GPU 0 handle 393 data.
  0%|          | 0/393 [00:00<?, ?it/s][ WARN:0@6.195] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
24 14:34:41 Save the image 01402D.png
 20%|█▉        | 77/393 [00:09<00:32,  9.76it/s]24 14:34:49 Save the image 01479D.png


 41%|████▏     | 163/393 [00:18<00:23,  9.63it/s]24 14:34:58 Save the image 01565D.png


 61%|██████    | 238/393 [00:26<00:15,  9.86it/s]24 14:35:06 Save the image 01222N.png
24 14:35:06 Save the image 01223N.png
 64%|██████▎   | 250/393 [00:27<00:14,  9.80it/s]24 14:35:07 Save the image 01234N.png


 65%|██████▌   | 257/393 [00:27<00:13,  9.87it/s]24 14:35:08 Save the image 01241N.png
24 14:35:08 Save the image 01242N.png
 66%|██████▌   | 259/393 [00:28<00:13,  9.93it/s]24 14:35:08 Save the image 01243N.png
24 14:35:08 Save the image 01244N.png
 67%|██████▋   | 262/393 [00:28<00:13,  9.98it/s]24 14:35:08 Save the image 01246N.png
24 14:35:08 Save the image 01247N.png
 68%|██████▊   | 269/393 [00:29<00:12,  9.85it/s]24 14:35:09 Save the image 01253N.png
24 14:35:09 Save the image 01254N.png
 69%|██████▉   | 273/393 [00:29<00:12,  9.90it/s]24 14:35:09 Save the image 01257N.png
24 14:35:09 Save the image 01258N.png
 86%|████████▋ | 339/393 [00:36<00:05,  9.64it/s]24 14:35:16 Save the image 01323N.png


100%|██████████| 393/393 [00:42<00:00,  9.31it/s]
24 14:35:22 Evaluation Elapsed Time: 42.20s


1 unlabeled	98.307%
2 car   	89.046%
3 person	75.508%
4 bike  	64.010%
5 curve 	47.270%
6 car_stop	28.214%
7 guardrail	6.214%
8 color_cone	52.291%
9 bump  	59.205%
----------     mean_IoU	57.785%	freq_IoU	96.756%	mean_pixel_acc	67.102%	pixel_acc	98.224%

Evaluation exit code: 0
Evaluation process completed. Capture the reported metrics.


In [25]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
PREDICTIONS = ROOT / "predictions_SCR_val"

assert (ROOT / "eval.py").is_file(), "eval.py not found"
assert (
    ROOT / "log_TDD_mit_b2_scr/checkpoint/epoch-100.pth"
).is_file(), "SCR epoch-100 checkpoint not found"

if PREDICTIONS.exists() and any(PREDICTIONS.iterdir()):
    raise RuntimeError(
        f"Prediction directory already contains files: {PREDICTIONS}\n"
        "Choose a fresh output directory to avoid mixing predictions."
    )

os.chdir(ROOT)

command = [
    "conda", "run", "--no-capture-output", "-n", "cmx",
    "python", "eval.py",
    "-e", "100",
    "-d", "0",
    "--split", "val",
    "-p", "predictions_SCR_val",
]

print("Evaluating SCR-CMX epoch 100 on the TDD validation split...")
result = subprocess.run(command, cwd=str(ROOT), check=False)

print("\nExit code:", result.returncode)
if result.returncode == 0:
    print("Validation evaluation completed. Share the metrics below.")
else:
    print("Evaluation failed. Share the error output.")

Evaluating SCR-CMX epoch 100 on the TDD validation split...


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 14:37:02 using devices 0
24 14:37:02 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
24 14:37:03 Using MLP Decoder
24 14:37:06 Load Model: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth
24 14:37:07 Load model, Time usage:
	IO: 0.7262370586395264, initialize parameters: 0.04177737236022949
24 14:37:07 GPU 0 handle 392 data.
  0%|          | 0/392 [00:00<?, ?it/s][ WARN:0@7.082] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
24 14:37:07 Save the image 00498D.png
 20%|█▉        | 77/392 [00:08<00:32,  9.70it/s]24 14:37:15 Save the image 00575D.png


 42%|████▏     | 163/392 [00:17<00:24,  9.36it/s]24 14:37:24 Save the image 00661D.png


 64%|██████▎   | 249/392 [00:26<00:14,  9.72it/s]24 14:37:33 Save the image 01046N.png


 85%|████████▌ | 335/392 [00:35<00:05,  9.52it/s]24 14:37:42 Save the image 01132N.png


100%|██████████| 392/392 [00:41<00:00,  9.56it/s]
24 14:37:48 Evaluation Elapsed Time: 41.03s


1 unlabeled	97.736%
2 car   	85.567%
3 person	72.279%
4 bike  	71.965%
5 curve 	39.565%
6 car_stop	36.822%
7 guardrail	46.395%
8 color_cone	58.726%
9 bump  	42.930%
----------     mean_IoU	61.332%	freq_IoU	95.648%	mean_pixel_acc	70.304%	pixel_acc	97.670%

Exit code: 0
Validation evaluation completed. Share the metrics below.


In [26]:

from pathlib import Path

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
LOG_DIR = ROOT / "log_TDD_mit_b2_scr"

print("=== SCR TRAINING FILES ===")

for path in sorted(LOG_DIR.rglob("*")):
    if path.is_file() and (
        path.suffix.lower() in {".log", ".txt", ".csv"}
        or "checkpoint" in str(path).lower()
    ):
        print(path)

print("\n=== CHECKPOINTS ===")
checkpoint_dir = LOG_DIR / "checkpoint"

if checkpoint_dir.exists():
    for path in sorted(checkpoint_dir.glob("*.pth")):
        print(f"{path.name}: {path.stat().st_size / (1024**3):.2f} GiB")
else:
    print("Checkpoint directory not found.")


=== SCR TRAINING FILES ===
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-10.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-15.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-20.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-25.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-30.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-35.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-40.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-45.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-5.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-50.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-55.pth
/home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoin

In [27]:

from pathlib import Path
import subprocess
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
CHECKPOINT = ROOT / "log_TDD_mit_b2_scr/checkpoint/epoch-100.pth"
PREDICTIONS = ROOT / "predictions_SCR_train"

assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"
assert (ROOT / "eval.py").is_file(), "eval.py not found"

if PREDICTIONS.exists() and any(PREDICTIONS.iterdir()):
    raise RuntimeError(
        f"Output directory already contains files: {PREDICTIONS}\n"
        "Use a new output directory to avoid mixing predictions."
    )

os.chdir(ROOT)

command = [
    "conda", "run", "--no-capture-output", "-n", "cmx",
    "python", "eval.py",
    "-e", "100",
    "-d", "0",
    "--split", "train",
    "-p", "predictions_SCR_train",
]

print("Evaluating SCR epoch 100 on the TDD training split...")
result = subprocess.run(command, cwd=str(ROOT), check=False)

print("\nExit code:", result.returncode)
if result.returncode == 0:
    print("Training-split evaluation completed. Share the metrics.")
else:
    print("Evaluation failed. Share the error output.")


Evaluating SCR epoch 100 on the TDD training split...


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 14:40:49 using devices 0
24 14:40:49 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
24 14:40:50 Using MLP Decoder
24 14:40:51 Load Model: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint/epoch-100.pth
24 14:40:51 Load model, Time usage:
	IO: 0.6610884666442871, initialize parameters: 0.04160308837890625
24 14:40:51 GPU 0 handle 1568 data.
  0%|          | 0/1568 [00:00<?, ?it/s][ WARN:0@6.805] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
24 14:40:54 Save the image 00001D.png
  5%|▍         | 74/1568 [00:09<02:33,  9.73it/s]24 14:41:01 Save the image 00114D.png


  5%|▌         | 83/1568 [00:10<02:32,  9.77it/s]24 14:41:02 Save the image 00118D_flip.png
24 14:41:02 Save the image 00119D.png
 10%|█         | 158/1568 [00:18<02:24,  9.79it/s]24 14:41:10 Save the image 00156D.png


 11%|█         | 169/1568 [00:19<02:22,  9.84it/s]24 14:41:11 Save the image 00161D_flip.png
24 14:41:11 Save the image 00162D.png
 11%|█▏        | 179/1568 [00:20<02:21,  9.85it/s]24 14:41:12 Save the image 00166D_flip.png
24 14:41:12 Save the image 00167D.png
 12%|█▏        | 184/1568 [00:21<02:19,  9.94it/s]24 14:41:13 Save the image 00169D.png
24 14:41:13 Save the image 00169D_flip.png
 15%|█▍        | 235/1568 [00:26<02:16,  9.78it/s]24 14:41:18 Save the image 00194D_flip.png
24 14:41:18 Save the image 00195D.png
 15%|█▌        | 237/1568 [00:26<02:15,  9.83it/s]24 14:41:18 Save the image 00195D_flip.png
24 14:41:18 Save the image 00196D.png
 16%|█▌        | 244/1568 [00:27<02:13,  9.89it/s]24 14:41:19 Save the image 00199D.png


 16%|█▌        | 247/1568 [00:27<02:12,  9.94it/s]24 14:41:19 Save the image 00200D_flip.png
24 14:41:19 Save the image 00201D.png
 17%|█▋        | 262/1568 [00:29<02:11,  9.90it/s]24 14:41:21 Save the image 00208D.png
24 14:41:21 Save the image 00208D_flip.png
 21%|██        | 325/1568 [00:35<02:06,  9.86it/s]24 14:41:27 Save the image 00239D_flip.png
24 14:41:27 Save the image 00240D.png
 21%|██        | 328/1568 [00:35<02:05,  9.91it/s]24 14:41:27 Save the image 00241D.png


...

 81%|████████  | 1272/1568 [02:12<00:30,  9.84it/s]24 14:43:04 Save the image 00854N.png
24 14:43:04 Save the image 00854N_flip.png
 82%|████████▏ | 1278/1568 [02:13<00:29,  9.93it/s]24 14:43:05 Save the image 00857N.png
24 14:43:05 Save the image 00857N_flip.png
 82%|████████▏ | 1281/1568 [02:13<00:28,  9.96it/s]24 14:43:05 Save the image 00858N_flip.png
24 14:43:05 Save the image 00859N.png
 82%|████████▏ | 1283/1568 [02:13<00:28,  9.98it/s]24 14:43:05 Save the image 00859N_flip.png
24 14:43:05 Save the image 00860N.png
 82%|████████▏ | 1289/1568 [02:14<00:28,  9.86it/s]24 14:43:06 Save the image 00862N_flip.png
24 14:43:06 Save the image 00863N.png
 82%|████████▏ | 1291/1568 [02:14<00:27,  9.92it/s]24 14:43:06 Save the image 00863N_flip.png
24 14:43:06 Save the image 00864N.png
 82%|████████▏ | 1293/1568 [02:14<00:27,  9.96it/s]24 14:43:06 Save the image 00864N_flip.png
24 14:43:06 Save the image 00865N.png
 83%|████████▎ | 1295/1568 [02:14<00:27,  9.99it/s]24 14:43:06 Save the imag

 91%|█████████ | 1427/1568 [02:28<00:14,  9.69it/s]24 14:43:20 Save the image 00931N_flip.png


 94%|█████████▎| 1468/1568 [02:32<00:10,  9.86it/s]24 14:43:24 Save the image 00952N.png
24 14:43:24 Save the image 00952N_flip.png
 95%|█████████▌| 1492/1568 [02:34<00:07,  9.89it/s]24 14:43:26 Save the image 00964N.png
24 14:43:27 Save the image 00964N_flip.png
 95%|█████████▌| 1497/1568 [02:35<00:07,  9.93it/s]24 14:43:27 Save the image 00966N_flip.png
24 14:43:27 Save the image 00967N.png
 96%|█████████▋| 1511/1568 [02:36<00:05,  9.85it/s]24 14:43:28 Save the image 00973N_flip.png


100%|██████████| 1568/1568 [02:42<00:00,  9.64it/s]
24 14:43:34 Evaluation Elapsed Time: 162.71s


1 unlabeled	99.789%
2 car   	98.728%
3 person	94.748%
4 bike  	95.660%
5 curve 	96.166%
6 car_stop	97.086%
7 guardrail	98.454%
8 color_cone	93.608%
9 bump  	97.980%
----------     mean_IoU	96.913%	freq_IoU	99.598%	mean_pixel_acc	98.520%	pixel_acc	99.797%

Exit code: 0
Training-split evaluation completed. Share the metrics.


# evaluation 

In [28]:
from pathlib import Path

roots = [
    Path("/home/cse-sdpl/Downloads/CMX_E5"),
    Path("/home/cse-sdpl/Downloads/CMX_SCR"),
]

for root in roots:
    print(f"\n{'=' * 70}\n{root}\n{'=' * 70}")

    if not root.exists():
        print("Directory not found")
        continue

    for pattern in [
        "*val*.log",
        "*eval*.log",
        "config.py",
        "eval.py",
    ]:
        matches = list(root.rglob(pattern))
        print(f"\n{pattern}:")
        for path in matches[-15:]:
            print(path)


/home/cse-sdpl/Downloads/CMX_E5

*val*.log:
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_23_18_56.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_23_18_11.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_19_23_29.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_19_11_50.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_23_15_24.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_19_09_42.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_last.log
/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/val_2026_10_16_19_07_46.log

*eval*.log:

config.py:
/home/cse-sdpl/Downloads/CMX_E5/config.py

eval.py:
/home/cse-sdpl/Downloads/CMX_E5/eval.py

/home/cse-sdpl/Downloads/CMX_SCR

*val*.log:
/home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/val_2026_10_16_23_18_11.log
/home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/val_2026_10_16_19_23_

In [29]:

from pathlib import Path
import re

ROOT = Path("/home/cse-sdpl/Downloads/CMX_E5")
LOG_DIR = ROOT / "log_TDD_mit_b2_e5"

log_files = sorted(LOG_DIR.glob("*val*.log"))

if not log_files:
    print("No validation logs found.")
else:
    for path in log_files:
        text = path.read_text(errors="replace")

        print("\n" + "=" * 85)
        print("FILE:", path.name)
        print("=" * 85)

        # Print metric lines
        metric_lines = [
            line.strip()
            for line in text.splitlines()
            if re.search(
                r"mean_IoU|freq_IoU|mean_pixel_acc|pixel_acc",
                line,
                re.IGNORECASE
            )
        ]

        if metric_lines:
            for line in metric_lines[-8:]:
                print(line)
        else:
            print("No standard metric lines found.")

        # Print class-wise IoUs if present
        class_lines = [
            line.strip()
            for line in text.splitlines()
            if re.search(
                r"unlabeled|car_stop|guardrail|color_cone",
                line,
                re.IGNORECASE
            )
        ]

        if class_lines:
            print("\nClass-related lines:")
            for line in class_lines[-12:]:
                print(line)



FILE: val_2026_10_16_19_07_46.log
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%

Class-related lines:
1 unlabeled	98.318%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%

FILE: val_2026_10_16_19_09_42.log
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%

Class-related lines:
1 unlabeled	98.318%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%

FILE: val_2026_10_16_19_11_50.log
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%

Class-related lines:
1 unlabeled	98.318%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%

FILE: val_2026_10_16_19_23_29.log
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%

Class-related lines:
1 unlabeled	98.318%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%

FILE: val_2026_10_16_23_15_24.log
----------     mean_IoU	96.611%	freq_IoU	99.557%	mean_pixel_a

In [30]:

from pathlib import Path
import re

log_dir = Path(
    "/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5"
)

targets = [
    "val_2026_10_16_23_15_24.log",
    "val_2026_10_16_23_18_11.log",
    "val_2026_10_16_23_18_56.log",
    "val_2026_10_16_19_11_50.log",
]

patterns = re.compile(
    r"split|train count|val count|test count|"
    r"checkpoint|predictions|mean_IoU|freq_IoU|"
    r"mean_pixel_acc|pixel_acc|evaluation|eval\.py|"
    r"dataset|epoch",
    re.IGNORECASE,
)

for name in targets:
    path = log_dir / name
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        (i, line.strip())
        for i, line in enumerate(lines)
        if patterns.search(line)
    ]

    for i, line in matches[:60]:
        print(f"L{i+1}: {line}")



val_2026_10_16_23_15_24.log
L1: Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
L11: ----------     mean_IoU	96.611%	freq_IoU	99.557%	mean_pixel_acc	98.424%	pixel_acc	99.776%

val_2026_10_16_23_18_11.log
L1: Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
L11: ----------     mean_IoU	61.947%	freq_IoU	95.732%	mean_pixel_acc	70.970%	pixel_acc	97.716%

val_2026_10_16_23_18_56.log
L1: Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
L11: ----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%

val_2026_10_16_19_11_50.log
L1: Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
L11: ----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%


In [31]:

from pathlib import Path

root = Path("/home/cse-sdpl/Downloads/CMX_E5")
eval_file = root / "eval.py"

lines = eval_file.read_text(errors="replace").splitlines()

keywords = [
    "split", "train.txt", "val.txt", "test.txt",
    "predictions", "argparse", "add_argument",
    "dataset", "log_file", "Model:"
]

for i, line in enumerate(lines):
    if any(k.lower() in line.lower() for k in keywords):
        start = max(0, i - 2)
        end = min(len(lines), i + 4)
        print(f"\n--- Lines {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")



--- Lines 2-7 ---
2: import os
3: import cv2
4: import argparse
5: import numpy as np
6: 
7: import torch

--- Lines 14-19 ---
14: from engine.logger import get_logger
15: from utils.metric import hist_info, compute_score
16: from dataloader.RGBXDataset import RGBXDataset
17: from models.builder import EncoderDecoder as segmodel
18: from dataloader.dataloader import ValPre
19: 

--- Lines 43-48 ---
43: 
44:         if self.show_image:
45:             colors = self.dataset.get_class_colors
46:             image = img
47:             clean = np.zeros(label.shape)
48:             comp_img = show_img(colors, config.background, image, clean,

--- Lines 67-72 ---
67:         iou, mean_IoU, _, freq_IoU, mean_pixel_acc, pixel_acc = compute_score(hist, correct, labeled)
68:         result_line = print_iou(iou, freq_IoU, mean_pixel_acc, pixel_acc,
69:                                 dataset.class_names, show_no_back=False)
70:         return result_line
71: 
72: if __name__ == "__main__":

--- 

In [32]:

from pathlib import Path
import subprocess
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_E5")
CHECKPOINT = ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"

assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"

os.chdir(ROOT)

for split in ["val", "test"]:
    output_dir = f"verified_E5_{split}_predictions"

    command = [
        "conda", "run", "--no-capture-output", "-n", "cmx",
        "python", "eval.py",
        "-e", "100",
        "-d", "0",
        "--split", split,
        "-p", output_dir,
    ]

    print("\n" + "=" * 70)
    print(f"Evaluating E5 epoch-100 on {split.upper()} split")
    print("=" * 70)

    result = subprocess.run(command, cwd=str(ROOT), check=False)
    print(f"{split.upper()} evaluation exit code: {result.returncode}")

    if result.returncode != 0:
        print(f"Evaluation failed for {split}. Stop and inspect the error.")
        break



Evaluating E5 epoch-100 on VAL split


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 14:50:31 using devices 0
24 14:50:31 Using E5: SegFormer-B2 with counterfactual reliability
24 14:50:32 Using MLP Decoder
24 14:50:32 Load Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
24 14:50:32 Load model, Time usage:
	IO: 0.6597409248352051, initialize parameters: 0.04165983200073242
24 14:50:32 GPU 0 handle 392 data.
  0%|          | 0/392 [00:00<?, ?it/s][ WARN:0@4.669] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
24 14:50:34 Save the image 00498D.png
  9%|▊         | 34/392 [00:04<00:36,  9.92it/s]24 14:50:37 Save the image 00532D.png
24 14:50:37 Save the image 00533D.png
 14%|█▍        | 5

 43%|████▎     | 168/392 [00:18<00:22,  9.75it/s]24 14:50:51 Save the image 00666D.png


 52%|█████▏    | 205/392 [00:22<00:18,  9.84it/s]24 14:50:55 Save the image 01002N.png
24 14:50:55 Save the image 01003N.png
 53%|█████▎    | 209/392 [00:22<00:18,  9.89it/s]24 14:50:55 Save the image 01006N.png
24 14:50:55 Save the image 01007N.png
 54%|█████▍    | 212/392 [00:22<00:18,  9.94it/s]24 14:50:56 Save the image 01009N.png
24 14:50:56 Save the image 01010N.png
 55%|█████▍    | 214/392 [00:23<00:17,  9.95it/s]24 14:50:56 Save the image 01011N.png
24 14:50:56 Save the image 01012N.png
 55%|█████▌    | 217/392 [00:23<00:17,  9.98it/s]24 14:50:56 Save the image 01014N.png
24 14:50:56 Save the image 01015N.png
 58%|█████▊    | 227/392 [00:24<00:16,  9.90it/s]24 14:50:57 Save the image 01024N.png
24 14:50:57 Save the image 01025N.png
 59%|█████▉    | 231/392 [00:24<00:16,  9.92it/s]24 14:50:57 Save the image 01028N.png
24 14:50:58 Save the image 01029N.png
 59%|█████▉    | 233/392 [00:25<00:15,  9.94it/s]24 14:50:58 Save the image 01030N.png
24 14:50:58 Save the image 01031N.png


 72%|███████▏  | 284/392 [00:30<00:10,  9.93it/s]24 14:51:03 Save the image 01081N.png
24 14:51:03 Save the image 01082N.png
 78%|███████▊  | 305/392 [00:32<00:08,  9.86it/s]24 14:51:05 Save the image 01102N.png
24 14:51:05 Save the image 01103N.png
 79%|███████▉  | 309/392 [00:32<00:08,  9.88it/s]24 14:51:05 Save the image 01106N.png
24 14:51:05 Save the image 01107N.png
 79%|███████▉  | 311/392 [00:32<00:08,  9.79it/s]24 14:51:06 Save the image 01108N.png
24 14:51:06 Save the image 01109N.png
 80%|████████  | 314/392 [00:33<00:07,  9.84it/s]24 14:51:06 Save the image 01111N.png
24 14:51:06 Save the image 01112N.png
 81%|████████  | 318/392 [00:33<00:07,  9.90it/s]24 14:51:06 Save the image 01115N.png
24 14:51:06 Save the image 01116N.png
 83%|████████▎ | 324/392 [00:34<00:06,  9.88it/s]24 14:51:07 Save the image 01121N.png
24 14:51:07 Save the image 01122N.png
 86%|████████▌ | 336/392 [00:35<00:05,  9.84it/s]24 14:51:08 Save the image 01133N.png
24 14:51:08 Save the image 01134N.png


...

 43%|████▎     | 168/393 [00:18<00:22,  9.94it/s]24 14:51:40 Save the image 01570D.png


 44%|████▎     | 171/393 [00:18<00:22,  9.94it/s]24 14:51:41 Save the image 01573D.png
24 14:51:41 Save the image 01574D.png
 44%|████▍     | 173/393 [00:18<00:22,  9.99it/s]24 14:51:41 Save the image 01575D.png
24 14:51:41 Save the image 01576D.png
 45%|████▍     | 175/393 [00:19<00:21, 10.00it/s]24 14:51:41 Save the image 01577D.png
24 14:51:41 Save the image 01578D.png
 50%|█████     | 197/393 [00:21<00:19,  9.89it/s]24 14:51:43 Save the image 01599D.png
24 14:51:43 Save the image 01600D.png
 52%|█████▏    | 205/393 [00:22<00:18,  9.95it/s]24 14:51:44 Save the image 01189N.png
24 14:51:44 Save the image 01190N.png
 53%|█████▎    | 207/393 [00:22<00:18, 10.01it/s]24 14:51:44 Save the image 01191N.png
24 14:51:44 Save the image 01192N.png
 53%|█████▎    | 209/393 [00:22<00:18, 10.05it/s]24 14:51:44 Save the image 01193N.png
24 14:51:45 Save the image 01194N.png
 54%|█████▎    | 211/393 [00:22<00:18, 10.08it/s]24 14:51:45 Save the image 01195N.png
24 14:51:45 Save the image 01196N.png


 72%|███████▏  | 282/393 [00:29<00:11,  9.45it/s]24 14:51:52 Save the image 01266N.png
24 14:51:52 Save the image 01267N.png
 72%|███████▏  | 284/393 [00:30<00:11,  9.71it/s]24 14:51:52 Save the image 01268N.png
24 14:51:52 Save the image 01269N.png
 73%|███████▎  | 286/393 [00:30<00:10,  9.85it/s]24 14:51:52 Save the image 01270N.png
24 14:51:52 Save the image 01271N.png
 73%|███████▎  | 288/393 [00:30<00:10,  9.90it/s]24 14:51:52 Save the image 01272N.png
24 14:51:53 Save the image 01273N.png
 74%|███████▍  | 290/393 [00:30<00:10,  9.95it/s]24 14:51:53 Save the image 01274N.png
24 14:51:53 Save the image 01275N.png
 75%|███████▍  | 294/393 [00:31<00:09,  9.91it/s]24 14:51:53 Save the image 01278N.png
24 14:51:53 Save the image 01279N.png
 76%|███████▌  | 298/393 [00:31<00:09,  9.97it/s]24 14:51:53 Save the image 01282N.png
24 14:51:54 Save the image 01283N.png
 76%|███████▋  | 300/393 [00:31<00:09,  9.99it/s]24 14:51:54 Save the image 01284N.png
24 14:51:54 Save the image 01285N.png


 95%|█████████▌| 374/393 [00:39<00:01,  9.98it/s]24 14:52:01 Save the image 01358N.png
24 14:52:01 Save the image 01359N.png
 96%|█████████▌| 376/393 [00:39<00:01,  9.99it/s]24 14:52:01 Save the image 01360N.png
24 14:52:01 Save the image 01361N.png
 96%|█████████▌| 378/393 [00:39<00:01,  9.99it/s]24 14:52:01 Save the image 01362N.png
24 14:52:02 Save the image 01363N.png
 97%|█████████▋| 380/393 [00:39<00:01, 10.02it/s]24 14:52:02 Save the image 01364N.png
24 14:52:02 Save the image 01365N.png
 97%|█████████▋| 382/393 [00:40<00:01, 10.02it/s]24 14:52:02 Save the image 01366N.png
24 14:52:02 Save the image 01367N.png
 98%|█████████▊| 384/393 [00:40<00:00, 10.02it/s]24 14:52:02 Save the image 01368N.png
24 14:52:02 Save the image 01369N.png
 98%|█████████▊| 386/393 [00:40<00:00, 10.03it/s]24 14:52:02 Save the image 01370N.png
24 14:52:02 Save the image 01371N.png
100%|██████████| 393/393 [00:41<00:00,  9.54it/s]
24 14:52:03 Evaluation Elapsed Time: 41.20s


1 unlabeled	98.318%
2 car   	89.264%
3 person	75.482%
4 bike  	65.006%
5 curve 	47.410%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%
9 bump  	58.436%
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
TEST evaluation exit code: 0


In [34]:

from pathlib import Path
import subprocess

ROOT = Path("/home/cse-sdpl/Downloads/CMX_E5")
CHECKPOINT = ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
OUTPUT_NAME = "verified_E5_val_predictions_run2"

assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"
assert (ROOT / "eval.py").is_file(), "eval.py not found"

command = [
    "conda", "run", "--no-capture-output", "-n", "cmx",
    "python", "eval.py",
    "-e", "100",
    "-d", "0",
    "--split", "val",
    "-p", OUTPUT_NAME,
]

print("Evaluating E5 epoch-100 on the VALIDATION split...")
print("Prediction directory:", ROOT / OUTPUT_NAME)

result = subprocess.run(command, cwd=str(ROOT), check=False)

print("\nExit code:", result.returncode)

if result.returncode == 0:
    print("Validation evaluation completed successfully, bro!")
else:
    print("Evaluation failed. Share the complete error output.")


Evaluating E5 epoch-100 on the VALIDATION split...
Prediction directory: /home/cse-sdpl/Downloads/CMX_E5/verified_E5_val_predictions_run2


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 14:54:31 using devices 0
24 14:54:31 Using E5: SegFormer-B2 with counterfactual reliability
24 14:54:32 Using MLP Decoder
24 14:54:34 Load Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
24 14:54:34 Load model, Time usage:
	IO: 0.68534255027771, initialize parameters: 0.0431520938873291
24 14:54:34 GPU 0 handle 392 data.
  0%|          | 0/392 [00:00<?, ?it/s][ WARN:0@7.191] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
24 14:54:36 Save the image 00498D.png
  4%|▍         | 16/392 [00:02<00:38,  9.77it/s]24 14:54:37 Save the image 00514D.png
24 14:54:38 Save the image 00515D.png
  7%|▋         | 27/3

 22%|██▏       | 86/392 [00:09<00:30,  9.92it/s]24 14:54:45 Save the image 00584D.png
24 14:54:45 Save the image 00585D.png
 23%|██▎       | 89/392 [00:10<00:30,  9.94it/s]24 14:54:45 Save the image 00587D.png
24 14:54:45 Save the image 00588D.png
 23%|██▎       | 92/392 [00:10<00:30,  9.96it/s]24 14:54:45 Save the image 00590D.png
24 14:54:45 Save the image 00591D.png
 24%|██▍       | 96/392 [00:10<00:29,  9.95it/s]24 14:54:46 Save the image 00594D.png
24 14:54:46 Save the image 00595D.png
 25%|██▌       | 98/392 [00:11<00:29,  9.97it/s]24 14:54:46 Save the image 00596D.png
24 14:54:46 Save the image 00597D.png
 26%|██▋       | 103/392 [00:11<00:29,  9.96it/s]24 14:54:46 Save the image 00601D.png
24 14:54:46 Save the image 00602D.png
 34%|███▍      | 133/392 [00:14<00:26,  9.93it/s]24 14:54:49 Save the image 00631D.png
24 14:54:49 Save the image 00632D.png
 35%|███▌      | 139/392 [00:15<00:25,  9.95it/s]24 14:54:50 Save the image 00637D.png
24 14:54:50 Save the image 00638D.png
 41%|

 46%|████▌     | 181/392 [00:19<00:21,  9.84it/s]24 14:54:54 Save the image 01378D.png
24 14:54:54 Save the image 01379D.png
 55%|█████▍    | 215/392 [00:23<00:19,  9.25it/s]24 14:54:58 Save the image 01012N.png
24 14:54:58 Save the image 01013N.png
 56%|█████▌    | 218/392 [00:23<00:17,  9.69it/s]24 14:54:58 Save the image 01015N.png
24 14:54:58 Save the image 01016N.png
 56%|█████▌    | 220/392 [00:23<00:17,  9.86it/s]24 14:54:58 Save the image 01017N.png
24 14:54:58 Save the image 01018N.png
 58%|█████▊    | 226/392 [00:24<00:16,  9.91it/s]24 14:54:59 Save the image 01023N.png
24 14:54:59 Save the image 01024N.png
 58%|█████▊    | 228/392 [00:24<00:16,  9.97it/s]24 14:54:59 Save the image 01025N.png
24 14:54:59 Save the image 01026N.png
 59%|█████▊    | 230/392 [00:24<00:16,  9.98it/s]24 14:54:59 Save the image 01027N.png
24 14:54:59 Save the image 01028N.png
 59%|█████▉    | 232/392 [00:24<00:15, 10.00it/s]24 14:54:59 Save the image 01029N.png
24 14:55:00 Save the image 01030N.png


24 14:55:03 Save the image 01064N.png
 68%|██████▊   | 268/392 [00:28<00:12,  9.96it/s]24 14:55:03 Save the image 01065N.png
24 14:55:03 Save the image 01066N.png
 69%|██████▉   | 272/392 [00:28<00:12,  9.97it/s]24 14:55:03 Save the image 01069N.png
24 14:55:04 Save the image 01070N.png
 70%|███████   | 275/392 [00:29<00:11,  9.95it/s]24 14:55:04 Save the image 01072N.png
24 14:55:04 Save the image 01073N.png
 71%|███████▏  | 280/392 [00:29<00:11,  9.88it/s]24 14:55:04 Save the image 01077N.png
24 14:55:04 Save the image 01078N.png
 72%|███████▏  | 283/392 [00:29<00:11,  9.90it/s]24 14:55:05 Save the image 01080N.png
24 14:55:05 Save the image 01081N.png
 73%|███████▎  | 285/392 [00:30<00:10,  9.92it/s]24 14:55:05 Save the image 01082N.png
24 14:55:05 Save the image 01083N.png
 74%|███████▍  | 290/392 [00:30<00:10,  9.89it/s]24 14:55:05 Save the image 01087N.png
24 14:55:05 Save the image 01088N.png
 76%|███████▌  | 296/392 [00:31<00:09,  9.86it/s]24 14:55:06 Save the image 01093N.png


 96%|█████████▋| 378/392 [00:39<00:01,  9.60it/s]24 14:55:14 Save the image 01175N.png
24 14:55:14 Save the image 01176N.png
 98%|█████████▊| 385/392 [00:40<00:00,  9.62it/s]24 14:55:15 Save the image 01182N.png
24 14:55:15 Save the image 01183N.png
100%|██████████| 392/392 [00:41<00:00,  9.52it/s]
24 14:55:16 Evaluation Elapsed Time: 41.18s


1 unlabeled	97.783%
2 car   	86.273%
3 person	72.271%
4 bike  	72.098%
5 curve 	39.137%
6 car_stop	36.476%
7 guardrail	47.055%
8 color_cone	61.487%
9 bump  	44.945%
----------     mean_IoU	61.947%	freq_IoU	95.732%	mean_pixel_acc	70.970%	pixel_acc	97.716%

Exit code: 0
Validation evaluation completed successfully, bro!


In [35]:
import subprocess
import os

print("=" * 90)
print("1. TRAINING PROCESSES RUNNING")
print("=" * 90)

result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,ppid,etime,%cpu,%mem,stat,args --sort=-%cpu"],
    capture_output=True,
    text=True
)

lines = result.stdout.splitlines()
keywords = [
    "train_crd.py",
    "train.py",
    "eval.py",
    "evaluate_crd.py",
    "CMX_E5",
    "CMX_E4",
    "CRD_Net",
]

matches = [
    line for line in lines
    if any(keyword.lower() in line.lower() for keyword in keywords)
    and "grep" not in line.lower()
    and "ps -eo" not in line.lower()
]

if matches:
    for line in matches:
        print(line)
else:
    print("No matching training/evaluation processes found.")

print("\n" + "=" * 90)
print("2. NVIDIA GPU STATUS")
print("=" * 90)

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,utilization.gpu,memory.used,memory.total",
        "--format=csv"
    ],
    capture_output=True,
    text=True
)

print(gpu.stdout if gpu.returncode == 0 else gpu.stderr)

print("\n" + "=" * 90)
print("3. CONDA PYTHON PROCESSES")
print("=" * 90)

result = subprocess.run(
    ["bash", "-lc", "pgrep -af 'python.*(train|eval|CRD_Net|CMX_E5|CMX_E4)'"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout.strip() else "No matching Python processes found.")

1. TRAINING PROCESSES RUNNING
 324364  324356       01:43  126 11.2 Rl   python /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2
 324459  324364       01:38  4.2  6.4 Sl   python /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2
 324458  324364       01:38  4.1  6.4 Sl   python /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2
 324355  317731       01:44  0.4  0.3 S    /home/cse-sdpl/miniconda3/bin/python /home/cse-sdpl/miniconda3/condabin/conda run --no-capture-output -n cmx python /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2

2. NVIDIA GPU STATUS
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
NVIDIA RTX 2000 Ada Generation, 100 %, 12843 MiB, 

In [36]:
from pathlib import Path

log_path = Path(
    "/home/cse-sdpl/Downloads/CRD_Net/logs/crd_full_training.log"
)

if log_path.exists():
    lines = log_path.read_text(errors="replace").splitlines()
    print("\n".join(lines[-25:]))
else:
    print("Training log not found:", log_path)

/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 17:48:44 Using E5: SegFormer-B2 with counterfactual reliability
24 17:48:45 Using MLP Decoder
24 17:48:45 Using E5: SegFormer-B2 with counterfactual reliability
24 17:48:46 Using MLP Decoder
CRD-Net initialization
  Device: cuda
  E5 checkpoint: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
  Checkpoint epoch: 100
  Loaded tensors: 869
  Missing keys (new CRD fusion parameters): 0
  Unexpected keys: 0
Resuming from epoch 5; checkpoint: /home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_epoch-last.pth
Existing best validation mIoU: 61.0326%
Validation images: 392

TRAINING MODE ENABLED: CRD outputs only.


In [37]:
import subprocess

pids = [273431, 317731, 325353]

for pid in pids:
    print(f"\n{'=' * 65}")
    print(f"PID {pid}")
    print("=" * 65)

    result = subprocess.run(
        [
            "bash", "-lc",
            f"ps -p {pid} -o pid,ppid,etime,%cpu,%mem,stat,args"
        ],
        capture_output=True,
        text=True
    )
    print(result.stdout.strip() or "Process no longer exists")


PID 273431
PID    PPID     ELAPSED %CPU %MEM STAT COMMAND
 273431    7279  1-00:22:41  0.3  5.3 Ssl  /home/cse-sdpl/miniconda3/envs/cmx/bin/python -m ipykernel_launcher -f /home/cse-sdpl/.local/share/jupyter/runtime/kernel-62bddaeb-8382-4a69-be5a-7ce1bee39534.json

PID 317731
PID    PPID     ELAPSED %CPU %MEM STAT COMMAND
 317731    7279    02:57:45  1.2 12.6 Ssl  /home/cse-sdpl/miniconda3/envs/cmx/bin/python -m ipykernel_launcher -f /home/cse-sdpl/.local/share/jupyter/runtime/kernel-f82aecea-8db8-4004-a67d-376aea031431.json

PID 325353
PID    PPID     ELAPSED %CPU %MEM STAT COMMAND
 325353  325345       06:26  107 11.3 Rl   python -u /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2


In [38]:
import os
import subprocess
import time

pid = 325353
log_path = "/home/cse-sdpl/Downloads/CRD_Net/logs/crd_full_training.log"

print("=" * 65)
print("CRD-NET PROCESS STATUS")
print("=" * 65)

result = subprocess.run(
    ["ps", "-p", str(pid), "-o", "pid,etime,%cpu,%mem,stat,args"],
    capture_output=True,
    text=True
)
print(result.stdout.strip() or "Training process is no longer running.")

print("\n" + "=" * 65)
print("LATEST TRAINING LOG")
print("=" * 65)

if os.path.exists(log_path):
    with open(log_path, "r", errors="replace") as f:
        lines = f.readlines()
    print("Log lines:", len(lines))
    print("".join(lines[-35:]) if lines else "Log exists but is empty.")
else:
    print("The background log file does not exist.")
    print("The current Jupyter cell may be displaying output directly instead.")

print("\n" + "=" * 65)
print("GPU STATUS")
print("=" * 65)

subprocess.run(["nvidia-smi"])

CRD-NET PROCESS STATUS
PID     ELAPSED %CPU %MEM STAT COMMAND
 325353       07:00  107 11.3 Rl   python -u /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2

LATEST TRAINING LOG
Log lines: 42
  Device: cuda
  E5 checkpoint: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
  Checkpoint epoch: 100
  Loaded tensors: 869
  Missing keys (new CRD fusion parameters): 0
  Unexpected keys: 0
Resuming from epoch 5; checkpoint: /home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_epoch-last.pth
Existing best validation mIoU: 61.0326%
Validation images: 392

TRAINING MODE ENABLED: CRD outputs only.
Terminated
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/multiprocessing/resource_tracker.py:224: UserWarning: resource_tracker: There appear to be 3 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
/home/cse-sdpl/miniconda3/envs/cmx/lib/py

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [39]:
import os
import subprocess
import time
from datetime import datetime

train_pid = 325353
checkpoint = "/home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_epoch-last.pth"
log_path = "/home/cse-sdpl/Downloads/CRD_Net/logs/crd_full_training.log"

print("=" * 65)
print("1. TRAINING PROCESS")
print("=" * 65)

result = subprocess.run(
    ["ps", "-p", str(train_pid), "-o", "pid,ppid,etime,%cpu,stat,args"],
    capture_output=True, text=True
)
print(result.stdout.strip() or "PID 325353 is no longer running.")

print("\n" + "=" * 65)
print("2. CHECKPOINT STATUS")
print("=" * 65)

if os.path.exists(checkpoint):
    stat = os.stat(checkpoint)
    print("Checkpoint:", checkpoint)
    print("Last modified:", datetime.fromtimestamp(stat.st_mtime))
    print("Size (MB):", round(stat.st_size / (1024 * 1024), 2))
else:
    print("Rolling checkpoint not found.")

print("\n" + "=" * 65)
print("3. LIVE LOG STATUS")
print("=" * 65)

if os.path.exists(log_path):
    stat = os.stat(log_path)
    print("Log last modified:", datetime.fromtimestamp(stat.st_mtime))
    with open(log_path, "r", errors="replace") as f:
        lines = f.readlines()
    print("Total lines:", len(lines))
    print("".join(lines[-20:]))
else:
    print("No background log found; the current run may stream directly to Jupyter.")

print("\n" + "=" * 65)
print("4. CRD TRAINING PROCESSES")
print("=" * 65)

subprocess.run([
    "bash", "-lc",
    "ps -eo pid,ppid,etime,%cpu,stat,args | grep '[t]rain_crd.py'"
])

1. TRAINING PROCESS
PID    PPID     ELAPSED %CPU STAT COMMAND
 325353  325345       08:07  106 Rl   python -u /home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py --run --resume --epochs 100 --steps-per-epoch 784 --batch-size 2 --workers 2

2. CHECKPOINT STATUS
Checkpoint: /home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_epoch-last.pth
Last modified: 2026-10-24 17:35:11.392127
Size (MB): 765.03

3. LIVE LOG STATUS
Log last modified: 2026-10-24 17:55:35.090704
Total lines: 42
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
24 17:54:45 Using E5: SegFormer-B2 with counterfactual reliability
24 17:54:46 Using MLP Decoder
24 17:54:46 Using E5: SegFormer-B2 with counterfactual reliability
24 17:54:47 Using MLP Decoder
CRD-Net initialization
  Device: cuda
  E5 checkpoint: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
  Checkpoint epoch: 100
  Loaded tensors: 869
  Missing keys (new CRD fusion parameters):

CompletedProcess(args=['bash', '-lc', "ps -eo pid,ppid,etime,%cpu,stat,args | grep '[t]rain_crd.py'"], returncode=0)

In [40]:

import os
import time
import signal
import subprocess
from pathlib import Path

SCRIPT = "/home/cse-sdpl/Downloads/CRD_Net/scripts/train_crd.py"

def find_crd_processes():
    result = subprocess.run(
        ["ps", "-eo", "pid=,args="],
        capture_output=True,
        text=True,
        check=True,
    )
    found = []
    for line in result.stdout.splitlines():
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        try:
            pid = int(parts[0])
        except ValueError:
            continue
        if SCRIPT in parts[1] and "train_crd.py" in parts[1]:
            if pid != os.getpid():
                found.append(pid)
    return sorted(set(found))

pids = find_crd_processes()
print("Existing CRD-Net training PIDs:", pids or "None")

for pid in pids:
    try:
        os.kill(pid, signal.SIGTERM)
        print("Sent stop signal:", pid)
    except ProcessLookupError:
        pass

for _ in range(15):
    time.sleep(1)
    remaining = find_crd_processes()
    if not remaining:
        break

remaining = find_crd_processes()
for pid in remaining:
    try:
        os.kill(pid, signal.SIGKILL)
        print("Force-stopped remaining CRD process:", pid)
    except ProcessLookupError:
        pass

time.sleep(2)
remaining = find_crd_processes()

if remaining:
    raise RuntimeError(f"CRD processes remain: {remaining}")

print("All CRD-Net training processes stopped.")
print("Ready for a fresh 100-epoch run.")


Existing CRD-Net training PIDs: [325344, 325353, 325447, 325448]
Sent stop signal: 325344
Sent stop signal: 325353
Sent stop signal: 325447
Sent stop signal: 325448
All CRD-Net training processes stopped.
Ready for a fresh 100-epoch run.


In [41]:

import os
import shutil
import subprocess
from pathlib import Path

HOME = Path.home()
DOWNLOADS = HOME / "Downloads"

print("=" * 75)
print("1. DISK SPACE")
print("=" * 75)

subprocess.run(["df", "-h", str(HOME)])

print("\n" + "=" * 75)
print("2. LARGEST DIRECTORIES IN DOWNLOADS")
print("=" * 75)

if DOWNLOADS.exists():
    result = subprocess.run(
        ["du", "-x", "-h", "--max-depth=1", str(DOWNLOADS)],
        capture_output=True, text=True
    )
    sizes = [
        line for line in result.stdout.splitlines()
        if line.strip()
    ]
    print("\n".join(sorted(
        sizes,
        key=lambda line: float(line.split()[0][:-1])
        if line.split()[0][-1].isdigit() is False
        else float(line.split()[0]),
        reverse=True
    )))
else:
    print("Downloads directory not found:", DOWNLOADS)

print("\n" + "=" * 75)
print("3. CRD-NET CHECKPOINTS AND LOGS")
print("=" * 75)

CRD = DOWNLOADS / "CRD_Net"

for folder in [
    CRD / "checkpoints",
    CRD / "logs",
]:
    if folder.exists():
        print(f"\n{folder}")
        subprocess.run(["du", "-sh", str(folder)])

        for file in sorted(
            folder.iterdir(),
            key=lambda p: p.stat().st_size if p.is_file() else 0,
            reverse=True
        ):
            if file.is_file():
                print(
                    f"{file.stat().st_size / (1024**2):9.2f} MB  "
                    f"{file.name}"
                )

print("\n" + "=" * 75)
print("4. CACHES")
print("=" * 75)

cache_paths = [
    HOME / ".cache/pip",
    HOME / ".cache/torch",
    HOME / ".cache/huggingface",
    HOME / ".cache/thumbnails",
    HOME / ".local/share/Trash/files",
]

for path in cache_paths:
    if path.exists():
        print(f"\n{path}")
        subprocess.run(["du", "-sh", str(path)])


1. DISK SPACE
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n1p5  438G  399G   18G  96% /

2. LARGEST DIRECTORIES IN DOWNLOADS
878M	/home/cse-sdpl/Downloads/segformers-20261011T205057Z-1-001
767M	/home/cse-sdpl/Downloads/MFNet_CMX+Segformer-B2_59.0
736M	/home/cse-sdpl/Downloads/ir_seg_dataset-20261011T201457Z-1-001 (2)
536K	/home/cse-sdpl/Downloads/cmx_sat_repo
203G	/home/cse-sdpl/Downloads
76M	/home/cse-sdpl/Downloads/ir_det_dataset
63M	/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARISON
48G	/home/cse-sdpl/Downloads/CMX_SCR
46G	/home/cse-sdpl/Downloads/CMX_E3
45M	/home/cse-sdpl/Downloads/anydesk-8.1.0-arm64
36K	/home/cse-sdpl/Downloads/CMX_Research_Matrices
29G	/home/cse-sdpl/Downloads/CMX_E2
17G	/home/cse-sdpl/Downloads/CMX_Original
16G	/home/cse-sdpl/Downloads/CMX_E4
16G	/home/cse-sdpl/Downloads/CMX_E5
14G	/home/cse-sdpl/Downloads/CMX_E1
9.6M	/home/cse-sdpl/Downloads/real-time-vehicle-counting-yolov8-botsort-main
6.0G	/home/cse-sdpl/Downloads/CRD_Net
4.0M	/home/cse-sdpl/D

In [42]:

import subprocess
from pathlib import Path

cache = Path.home() / ".cache/huggingface"

print("HUGGING FACE CACHE BREAKDOWN")
print("=" * 65)

if cache.exists():
    subprocess.run([
        "du", "-h", "--max-depth=2", str(cache)
    ])

hub = cache / "hub"

if hub.exists():
    print("\nLargest cached model repositories:")
    subprocess.run([
        "du", "-sh", *[str(p) for p in hub.iterdir()]
    ], check=False)


HUGGING FACE CACHE BREAKDOWN
7.1G	/home/cse-sdpl/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct
12K	/home/cse-sdpl/.cache/huggingface/hub/datasets--AI4Math--MathVista
11G	/home/cse-sdpl/.cache/huggingface/hub/models--microsoft--Phi-4-multimodal-instruct
28K	/home/cse-sdpl/.cache/huggingface/hub/.locks
533M	/home/cse-sdpl/.cache/huggingface/hub/models--microsoft--deberta-base-mnli
18G	/home/cse-sdpl/.cache/huggingface/hub
8.0K	/home/cse-sdpl/.cache/huggingface/xet/https___cas_serv-tGqkUaZf_CBPHQ6h
17M	/home/cse-sdpl/.cache/huggingface/xet/logs
17M	/home/cse-sdpl/.cache/huggingface/xet
792K	/home/cse-sdpl/.cache/huggingface/modules/transformers_modules
796K	/home/cse-sdpl/.cache/huggingface/modules
18G	/home/cse-sdpl/.cache/huggingface

Largest cached model repositories:
7.1G	/home/cse-sdpl/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct
12K	/home/cse-sdpl/.cache/huggingface/hub/datasets--AI4Math--MathVista
11G	/home/cse-sdpl/.cache/huggingface/hub/models--micros

In [43]:

from pathlib import Path
import subprocess

base = Path("/home/cse-sdpl/Downloads")

folders = [
    "CMX_SCR",
    "CMX_E1",
    "CMX_E2",
    "CMX_E3",
    "CMX_E4",
    "CMX_E5",
    "CMX_Original",
]

for name in folders:
    folder = base / name
    if not folder.exists():
        continue

    print("\n" + "=" * 70)
    print(f"{name} — largest files")
    print("=" * 70)

    result = subprocess.run(
        [
            "bash", "-lc",
            f"find '{folder}' -type f -size +500M "
            "-printf '%s %p\\n' 2>/dev/null | sort -nr | head -15"
        ],
        capture_output=True,
        text=True,
    )

    for line in result.stdout.splitlines():
        parts = line.split(" ", 1)
        if len(parts) == 2:
            size_gb = int(parts[0]) / (1024 ** 3)
            print(f"{size_gb:7.2f} GB  {parts[1]}")



CMX_SCR — largest files
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/pretraining_backups/e4_epoch-100_initialization.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/checkpoint/epoch-last.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/pretraining_backups/e4_epoch-100_initialization.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint/epoch-last.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/checkpoint/epoch-95.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/checkpoint/epoch-90.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_mit_b2_e5/checkpoint/epoch-85.pth
   0.75 GB  /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/log_TDD_m

In [44]:
from pathlib import Path
import shutil

cache = Path.home() / ".cache" / "huggingface" / "hub"

targets = [
    cache / "models--microsoft--Phi-4-multimodal-instruct",
    cache / "models--Qwen--Qwen2.5-VL-3B-Instruct",
]

for path in targets:
    if path.exists():
        size_gb = sum(
            f.stat().st_size for f in path.rglob("*") if f.is_file()
        ) / (1024 ** 3)
        print(f"{path.name}: {size_gb:.2f} GB")
    else:
        print(f"Not found: {path.name}")

confirm = input(
    "\nType DELETE-CACHES to permanently delete these two cached models: "
).strip()

if confirm == "DELETE-CACHES":
    for path in targets:
        if path.exists():
            shutil.rmtree(path)
            print(f"Deleted: {path.name}")
    print("\nCleanup complete.")
else:
    print("\nCancelled. No files were deleted.")

models--microsoft--Phi-4-multimodal-instruct: 20.81 GB
models--Qwen--Qwen2.5-VL-3B-Instruct: 14.01 GB

Type DELETE-CACHES to permanently delete these two cached models: DELETE-CACHES
Deleted: models--microsoft--Phi-4-multimodal-instruct
Deleted: models--Qwen--Qwen2.5-VL-3B-Instruct

Cleanup complete.


In [45]:
import shutil

usage = shutil.disk_usage("/")

print(f"Total disk: {usage.total / 1024**3:.1f} GB")
print(f"Free space: {usage.free / 1024**3:.1f} GB")
print(f"Disk usage: {(usage.total - usage.free) / usage.total * 100:.1f}%")

Total disk: 437.9 GB
Free space: 34.5 GB
Disk usage: 92.1%


In [46]:
from pathlib import Path
import re

log_file = Path(
    "/home/cse-sdpl/Downloads/CRD_Net/logs/"
    "crd_fresh_100_epochs_20261024_181703.log"
)

if not log_file.exists():
    print("Log file not found:", log_file)
else:
    text = log_file.read_text(errors="ignore")

    print("LATEST EPOCH TIMINGS / PROGRESS")
    print("=" * 65)

    lines = text.splitlines()

    for line in lines[-150:]:
        if re.search(
            r"epoch|Epoch|training|Training|validation|Validation|"
            r"took|elapsed|remaining|ETA|it/s|s/it|mIoU|mean_IoU",
            line,
            re.IGNORECASE,
        ):
            print(line)

LATEST EPOCH TIMINGS / PROGRESS
100%|██████████| 392/392 [00:43<00:00,  9.05it/s]
\nCRD-Net validation metrics
Validation mIoU: 60.5196%
24 18:43:45 Evaluation Elapsed Time: 43.32s
Epoch 2 validation mIoU=60.5196%
Saved new best checkpoint: /home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_best_miou.pth (mIoU=60.5196%)


In [47]:
from pathlib import Path
import re

log = Path(
    "/home/cse-sdpl/Downloads/CRD_Net/logs/"
    "crd_fresh_100_epochs_20261024_181703.log"
)

lines = log.read_text(errors="ignore").splitlines()

for line in lines:
    if re.search(
        r"Epoch\s+\d+|epoch\s+\d+|Training Elapsed|"
        r"Evaluation Elapsed|validation mIoU|Saved new best",
        line,
        re.IGNORECASE,
    ):
        print(line)

Epoch 1/100 | lr=6.00e-05 | steps=784 | total=0.080362 | seg=0.010343 | reliability=0.690490 | distillation=0.009695
Validation mIoU: 59.2429%
24 18:30:27 Evaluation Elapsed Time: 43.07s
Epoch 1 validation mIoU=59.2429%
Saved new best checkpoint: /home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_best_miou.pth (mIoU=59.2429%)
Epoch 2/100 | lr=6.00e-05 | steps=784 | total=0.081703 | seg=0.011609 | reliability=0.687368 | distillation=0.013574
Validation mIoU: 60.5196%
24 18:43:45 Evaluation Elapsed Time: 43.32s
Epoch 2 validation mIoU=60.5196%
Saved new best checkpoint: /home/cse-sdpl/Downloads/CRD_Net/checkpoints/crd_best_miou.pth (mIoU=60.5196%)
